In [81]:
import zipfile

zip_path = "Indian-Railway-Network-and-Delays.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    files = zip_ref.namelist()

print("Files inside ZIP:")
for f in files:
    print(f)

Files inside ZIP:
Indian-Railway-Network-and-Delays/
Indian-Railway-Network-and-Delays/train_routes_Sep2024.csv
Indian-Railway-Network-and-Delays/IRN_edges.csv
Indian-Railway-Network-and-Delays/train_routes_delays_Sep2024.csv
Indian-Railway-Network-and-Delays/stations_zones_mapping.json
Indian-Railway-Network-and-Delays/train_delays_Sep2024.json


In [82]:
import zipfile

zip_path = "Indian-Railway-Network-and-Delays.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    file_to_extract = "Indian-Railway-Network-and-Delays/train_routes_delays_Sep2024.csv"
    zip_ref.extract(file_to_extract, ".")

print("Extracted successfully!")

Extracted successfully!


In [83]:
import pandas as pd

delay_df = pd.read_csv(
    "Indian-Railway-Network-and-Delays/train_routes_delays_Sep2024.csv"
)

print("Shape:", delay_df.shape)
print("\nColumns:")
print(delay_df.columns.tolist())

print("\nFirst 5 rows:")
display(delay_df.head())

Shape: (1282325, 9)

Columns:
['train', 'date', 'station', 'sch_arr', 'act_arr', 'arr_delay', 'sch_dep', 'act_dep', 'dep_delay']

First 5 rows:


,train,date,station,sch_arr,act_arr,arr_delay,sch_dep,act_dep,dep_delay
0,12303,2024-09-02,HWH,08:00 AM,08:01 AM,0.0,08:00 AM,08:01 AM,0.0
1,12303,2024-09-02,BWN,09:05 AM,09:20 AM,15.0,09:08 AM,09:23 AM,15.0
2,12303,2024-09-02,DGR,09:57 AM,10:00 AM,3.0,09:59 AM,10:02 AM,3.0
3,12303,2024-09-02,ASN,10:32 AM,10:34 AM,2.0,10:37 AM,10:39 AM,2.0
4,12303,2024-09-02,CRJ,11:00 AM,11:04 AM,4.0,11:02 AM,11:06 AM,4.0


In [84]:
print("Shape:", delay_df.shape)

print("\nData types:")
print(delay_df.dtypes)

print("\nMissing values:")
print(delay_df.isnull().sum())

print("\nUnique trains:", delay_df["train"].nunique())
print("Unique dates:", delay_df["date"].nunique())
print("Unique stations:", delay_df["station"].nunique())

Shape: (1282325, 9)

Data types:
train          int64
date          object
station       object
sch_arr       object
act_arr       object
arr_delay    float64
sch_dep       object
act_dep       object
dep_delay    float64
dtype: object

Missing values:
train        0
date         0
station      0
sch_arr      0
act_arr      0
arr_delay    0
sch_dep      0
act_dep      0
dep_delay    0
dtype: int64

Unique trains: 3892
Unique dates: 30
Unique stations: 4736


In [85]:
# Sort by train and date so stations are in journey order
delay_df = delay_df.sort_values(
    ["train", "date"]
).reset_index(drop=True)

# Create the next station and next-station delays
delay_df["next_station"] = delay_df.groupby(
    ["train", "date"]
)["station"].shift(-1)

delay_df["next_arr_delay"] = delay_df.groupby(
    ["train", "date"]
)["arr_delay"].shift(-1)

delay_df["next_dep_delay"] = delay_df.groupby(
    ["train", "date"]
)["dep_delay"].shift(-1)

# Remove the final station of each train journey
next_station_df = delay_df.dropna(
    subset=["next_station", "next_arr_delay"]
).copy()

print("Original rows:", len(delay_df))
print("Next-station rows:", len(next_station_df))

display(
    next_station_df[
        [
            "train",
            "date",
            "station",
            "next_station",
            "arr_delay",
            "next_arr_delay",
            "dep_delay",
            "next_dep_delay"
        ]
    ].head(10)
)

Original rows: 1282325
Next-station rows: 1224840


,train,date,station,next_station,arr_delay,next_arr_delay,dep_delay,next_dep_delay
0,961,2024-09-05,MJ,FLD,0.0,0.0,0.0,0.0
1,961,2024-09-05,FLD,KBK,0.0,0.0,0.0,0.0
3,961,2024-09-11,MJ,FLD,0.0,58.0,0.0,58.0
4,961,2024-09-11,FLD,KBK,58.0,35.0,58.0,35.0
6,961,2024-09-12,MJ,FLD,3.0,0.0,3.0,0.0
7,961,2024-09-12,FLD,KBK,0.0,0.0,0.0,0.0
9,961,2024-09-14,MJ,FLD,3.0,4.0,3.0,4.0
10,961,2024-09-14,FLD,KBK,4.0,14.0,4.0,14.0
12,961,2024-09-15,MJ,FLD,0.0,0.0,0.0,0.0
13,961,2024-09-15,FLD,KBK,0.0,0.0,0.0,0.0


In [86]:
# Check one complete train journey
sample_train = 961
sample_date = "2024-09-11"

sample = delay_df[
    (delay_df["train"] == sample_train) &
    (delay_df["date"] == sample_date)
]

print(sample[
    ["train", "date", "station", "arr_delay"]
].to_string(index=False))

 train       date station  arr_delay
   961 2024-09-11      MJ        0.0
   961 2024-09-11     FLD       58.0
   961 2024-09-11     KBK       35.0


In [87]:
# Check whether any train/date journey has duplicate stations
duplicates = delay_df.duplicated(
    subset=["train", "date", "station"]
).sum()

print("Duplicate train-date-station rows:", duplicates)

Duplicate train-date-station rows: 0


In [88]:
# Check journey lengths
journey_lengths = delay_df.groupby(
    ["train", "date"]
).size()

print("Number of train journeys:", len(journey_lengths))
print("\nJourney length statistics:")
print(journey_lengths.describe())

Number of train journeys: 57485

Journey length statistics:
count    57485.000000
mean        22.307124
std         14.581407
min          2.000000
25%         12.000000
50%         19.000000
75%         29.000000
max        119.000000
dtype: float64


In [89]:
next_station_df = next_station_df.rename(columns={
    "station": "current_station",
    "arr_delay": "current_arr_delay",
    "dep_delay": "current_dep_delay",
    "next_arr_delay": "target_next_arr_delay"
})

print(next_station_df.columns.tolist())

['train', 'date', 'current_station', 'sch_arr', 'act_arr', 'current_arr_delay', 'sch_dep', 'act_dep', 'current_dep_delay', 'next_station', 'target_next_arr_delay', 'next_dep_delay']


In [90]:
next_station_df["date"] = pd.to_datetime(next_station_df["date"])

next_station_df["year"] = next_station_df["date"].dt.year
next_station_df["month"] = next_station_df["date"].dt.month
next_station_df["day_of_week"] = next_station_df["date"].dt.dayofweek
next_station_df["is_weekend"] = (
    next_station_df["day_of_week"] >= 5
).astype(int)

print(
    next_station_df[
        ["date", "year", "month", "day_of_week", "is_weekend"]
    ].head()
)

        date  year  month  day_of_week  is_weekend
0 2024-09-05  2024      9            3           0
1 2024-09-05  2024      9            3           0
3 2024-09-11  2024      9            2           0
4 2024-09-11  2024      9            2           0
6 2024-09-12  2024      9            3           0


In [91]:
# Get the next station's scheduled arrival time
delay_df["next_sch_arr"] = delay_df.groupby(
    ["train", "date"]
)["sch_arr"].shift(-1)

# Add it to our next-station dataframe
next_station_df["next_sch_arr"] = delay_df.loc[
    next_station_df.index, "next_sch_arr"
]

print(
    next_station_df[
        ["train", "date", "current_station", "next_station",
         "sch_arr", "next_sch_arr"]
    ].head(10)
)

    train       date current_station next_station   sch_arr next_sch_arr
0     961 2024-09-05              MJ          FLD  09:45 AM     11:00 AM
1     961 2024-09-05             FLD          KBK  11:00 AM     12:45 PM
3     961 2024-09-11              MJ          FLD  09:45 AM     11:00 AM
4     961 2024-09-11             FLD          KBK  11:00 AM     12:45 PM
6     961 2024-09-12              MJ          FLD  09:45 AM     11:00 AM
7     961 2024-09-12             FLD          KBK  11:00 AM     12:45 PM
9     961 2024-09-14              MJ          FLD  09:45 AM     11:00 AM
10    961 2024-09-14             FLD          KBK  11:00 AM     12:45 PM
12    961 2024-09-15              MJ          FLD  09:45 AM     11:00 AM
13    961 2024-09-15             FLD          KBK  11:00 AM     12:45 PM


In [92]:
# Convert scheduled arrival times to datetime
sch_current = pd.to_datetime(
    next_station_df["sch_arr"],
    format="%I:%M %p"
)

sch_next = pd.to_datetime(
    next_station_df["next_sch_arr"],
    format="%I:%M %p"
)

# Convert to minutes since midnight
next_station_df["sch_arr_minutes"] = (
    sch_current.dt.hour * 60 + sch_current.dt.minute
)

next_station_df["next_sch_arr_minutes"] = (
    sch_next.dt.hour * 60 + sch_next.dt.minute
)

# Calculate scheduled travel time
next_station_df["scheduled_segment_minutes"] = (
    next_station_df["next_sch_arr_minutes"]
    - next_station_df["sch_arr_minutes"]
)

print(
    next_station_df[
        [
            "train",
            "current_station",
            "next_station",
            "sch_arr",
            "next_sch_arr",
            "scheduled_segment_minutes"
        ]
    ].head(10)
)


    train current_station next_station   sch_arr next_sch_arr  \
0     961              MJ          FLD  09:45 AM     11:00 AM   
1     961             FLD          KBK  11:00 AM     12:45 PM   
3     961              MJ          FLD  09:45 AM     11:00 AM   
4     961             FLD          KBK  11:00 AM     12:45 PM   
6     961              MJ          FLD  09:45 AM     11:00 AM   
7     961             FLD          KBK  11:00 AM     12:45 PM   
9     961              MJ          FLD  09:45 AM     11:00 AM   
10    961             FLD          KBK  11:00 AM     12:45 PM   
12    961              MJ          FLD  09:45 AM     11:00 AM   
13    961             FLD          KBK  11:00 AM     12:45 PM   

    scheduled_segment_minutes  
0                          75  
1                         105  
3                          75  
4                         105  
6                          75  
7                         105  
9                          75  
10                        10

In [93]:
# Check for negative scheduled segment times
negative_segments = (
    next_station_df["scheduled_segment_minutes"] < 0
).sum()

print("Negative segment times:", negative_segments)

# Show a few if they exist
if negative_segments > 0:
    display(
        next_station_df[
            next_station_df["scheduled_segment_minutes"] < 0
        ][
            [
                "train",
                "date",
                "current_station",
                "next_station",
                "sch_arr",
                "next_sch_arr",
                "scheduled_segment_minutes"
            ]
        ].head(10)
    )

Negative segment times: 39444


,train,date,current_station,next_station,sch_arr,next_sch_arr,scheduled_segment_minutes
58,1007,2024-09-06,TIR,SRR,11:14 PM,12:10 AM,-1384
70,1008,2024-09-07,VLNK,NGT,11:55 PM,12:10 AM,-1425
103,1023,2024-09-01,WTR,STR,11:53 PM,12:22 AM,-1411
125,1023,2024-09-02,WTR,STR,11:53 PM,12:22 AM,-1411
147,1023,2024-09-03,WTR,STR,11:53 PM,12:22 AM,-1411
169,1023,2024-09-04,WTR,STR,11:53 PM,12:22 AM,-1411
191,1023,2024-09-05,WTR,STR,11:53 PM,12:22 AM,-1411
213,1023,2024-09-06,WTR,STR,11:53 PM,12:22 AM,-1411
235,1023,2024-09-08,WTR,STR,11:53 PM,12:22 AM,-1411
257,1023,2024-09-09,WTR,STR,11:53 PM,12:22 AM,-1411


In [94]:
# Fix segments that cross midnight
next_station_df.loc[
    next_station_df["scheduled_segment_minutes"] < 0,
    "scheduled_segment_minutes"
] += 24 * 60

print("Negative segments after fix:",
      (next_station_df["scheduled_segment_minutes"] < 0).sum())

print(
    next_station_df[
        next_station_df["scheduled_segment_minutes"] >= 0
    ][[
        "train",
        "current_station",
        "next_station",
        "sch_arr",
        "next_sch_arr",
        "scheduled_segment_minutes"
    ]].head()
)

Negative segments after fix: 0
   train current_station next_station   sch_arr next_sch_arr  \
0    961              MJ          FLD  09:45 AM     11:00 AM   
1    961             FLD          KBK  11:00 AM     12:45 PM   
3    961              MJ          FLD  09:45 AM     11:00 AM   
4    961             FLD          KBK  11:00 AM     12:45 PM   
6    961              MJ          FLD  09:45 AM     11:00 AM   

   scheduled_segment_minutes  
0                         75  
1                        105  
3                         75  
4                        105  
6                         75  


In [95]:
print("Target statistics:")
print(next_station_df["target_next_arr_delay"].describe())

print("\nCurrent delay statistics:")
print(next_station_df["current_arr_delay"].describe())

print("\nCorrelation:")
print(
    next_station_df[
        ["current_arr_delay", "current_dep_delay",
         "scheduled_segment_minutes", "target_next_arr_delay"]
    ].corr()
)

Target statistics:
count    1.224840e+06
mean     3.391143e+01
std      8.555710e+01
min      0.000000e+00
25%      0.000000e+00
50%      9.000000e+00
75%      3.000000e+01
max      4.336000e+03
Name: target_next_arr_delay, dtype: float64

Current delay statistics:
count    1.224840e+06
mean     3.321698e+01
std      8.409976e+01
min      0.000000e+00
25%      0.000000e+00
50%      8.000000e+00
75%      2.900000e+01
max      4.336000e+03
Name: current_arr_delay, dtype: float64

Correlation:
                           current_arr_delay  current_dep_delay  \
current_arr_delay                   1.000000           1.000000   
current_dep_delay                   1.000000           1.000000   
scheduled_segment_minutes           0.097573           0.097573   
target_next_arr_delay               0.922205           0.922205   

                           scheduled_segment_minutes  target_next_arr_delay  
current_arr_delay                           0.097573               0.922205  
current_dep_

In [96]:
print("Date range:")
print(next_station_df["date"].min())
print(next_station_df["date"].max())

print("\nRows by date:")
print(next_station_df["date"].value_counts().sort_index())

Date range:
2024-09-01 00:00:00
2024-09-30 00:00:00

Rows by date:
date
2024-09-01    38768
2024-09-02    38976
2024-09-03    39574
2024-09-04    39879
2024-09-05    39823
2024-09-06    40900
2024-09-07    39756
2024-09-08    41056
2024-09-09    41034
2024-09-10    40209
2024-09-11    40034
2024-09-12    40416
2024-09-13    41371
2024-09-14    40865
2024-09-15    41187
2024-09-16    42598
2024-09-17    42459
2024-09-18    41793
2024-09-19    41493
2024-09-20    41986
2024-09-21    41354
2024-09-22    41229
2024-09-23    41198
2024-09-24    40423
2024-09-25    40395
2024-09-26    41500
2024-09-27    42137
2024-09-28    41074
2024-09-29    40501
2024-09-30    40852
Name: count, dtype: int64


In [97]:
train_ns = next_station_df[
    next_station_df["date"] < "2024-09-24"
].copy()

val_ns = next_station_df[
    next_station_df["date"] >= "2024-09-24"
].copy()

print("Training shape:", train_ns.shape)
print("Validation shape:", val_ns.shape)

print("\nTraining dates:")
print(train_ns["date"].min(), "to", train_ns["date"].max())

print("\nValidation dates:")
print(val_ns["date"].min(), "to", val_ns["date"].max())

Training shape: (937958, 20)
Validation shape: (286882, 20)

Training dates:
2024-09-01 00:00:00 to 2024-09-23 00:00:00

Validation dates:
2024-09-24 00:00:00 to 2024-09-30 00:00:00


In [98]:
categorical_cols = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_cols:
    train_ns[col] = train_ns[col].astype("category")
    val_ns[col] = val_ns[col].astype("category")

print("Categorical columns:")
print(train_ns[categorical_cols].dtypes)

Categorical columns:
train              category
current_station    category
next_station       category
dtype: object


In [99]:
categorical_cols = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_cols:
    train_ns[col] = train_ns[col].astype("category")
    val_ns[col] = val_ns[col].astype("category")

print("Categorical columns:")
print(train_ns[categorical_cols].dtypes)

Categorical columns:
train              category
current_station    category
next_station       category
dtype: object


In [100]:
# Features for our first baseline model
model_features = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend"
]

target = "target_next_arr_delay"

X_train_ns = train_ns[model_features]
y_train_ns = train_ns[target]

X_val_ns = val_ns[model_features]
y_val_ns = val_ns[target]

print("X_train:", X_train_ns.shape)
print("y_train:", y_train_ns.shape)
print("X_val:", X_val_ns.shape)
print("y_val:", y_val_ns.shape)

X_train: (937958, 8)
y_train: (937958,)
X_val: (286882, 8)
y_val: (286882,)


In [101]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

model_ns = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_ns.fit(
    X_train_ns,
    y_train_ns,
    categorical_feature=categorical_cols
)

print("Model trained!")


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.100315 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11880
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 7
[LightGBM] [Info] Start training from score 34.667424
Model trained!


In [102]:
pred_ns = model_ns.predict(X_val_ns)

mae_ns = mean_absolute_error(y_val_ns, pred_ns)
rmse_ns = np.sqrt(mean_squared_error(y_val_ns, pred_ns))

print(f"Baseline Validation MAE: {mae_ns:.2f} minutes")
print(f"Baseline Validation RMSE: {rmse_ns:.2f} minutes")


Baseline Validation MAE: 9.14 minutes
Baseline Validation RMSE: 28.60 minutes


In [103]:
simple_pred = X_val_ns["current_arr_delay"].values

simple_mae = mean_absolute_error(y_val_ns, simple_pred)
simple_rmse = np.sqrt(
    mean_squared_error(y_val_ns, simple_pred)
)

print(f"Simple persistence MAE: {simple_mae:.2f} minutes")
print(f"Simple persistence RMSE: {simple_rmse:.2f} minutes")

Simple persistence MAE: 9.83 minutes
Simple persistence RMSE: 31.49 minutes


In [104]:
importance = pd.DataFrame({
    "feature": model_ns.feature_name_,
    "importance": model_ns.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance)

                     feature  importance
3          current_arr_delay        3890
0                      train        3567
2               next_station        3099
1            current_station        2745
4  scheduled_segment_minutes        1118
6                day_of_week         581
5                      month           0
7                 is_weekend           0


In [105]:
print(
    "Unique train numbers:",
    next_station_df["train"].nunique()
)

print(
    "Unique current stations:",
    next_station_df["current_station"].nunique()
)

print(
    "Unique next stations:",
    next_station_df["next_station"].nunique()
)

print(
    "Unique station pairs:",
    next_station_df[
        ["current_station", "next_station"]
    ].drop_duplicates().shape[0]
)

Unique train numbers: 3892
Unique current stations: 4728
Unique next stations: 4728
Unique station pairs: 17317


In [106]:
# Create a station-pair identifier
next_station_df["segment"] = (
    next_station_df["current_station"]
    + "->"
    + next_station_df["next_station"]
)

print("Unique segments:", next_station_df["segment"].nunique())

# Segment statistics
segment_stats = (
    next_station_df
    .groupby("segment")["target_next_arr_delay"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("count", ascending=False)
)

print(segment_stats.head(10))

Unique segments: 17317
               mean  median         std  count
segment                                       
DR->CSMT  18.926050     0.0   56.985143   1190
KUR->BBS  39.594737     9.0   99.518237   1140
JL->BSL   35.255526    13.0   83.936529   1131
BBS->KUR  70.667860    25.0  129.533806   1117
KYN->TNA  40.438739     7.0   89.022997   1110
CSMT->DR  19.239856     7.0   50.731955   1109
TNA->KYN  27.860000    16.0   50.629652   1100
BSL->JL   83.500928    12.0  177.184955   1078
TUP->ED   18.975928     8.0   48.946875    997
SA->ED    41.174975     8.0  138.442890    983


In [107]:
# Historical segment statistics from TRAINING data only

segment_history = (
    train_ns
    .groupby(["current_station", "next_station"])["target_next_arr_delay"]
    .agg(
        segment_mean_delay="mean",
        segment_median_delay="median",
        segment_std_delay="std",
        segment_count="count"
    )
    .reset_index()
)

print("Historical segment records:", len(segment_history))
print(segment_history.head(10))


/tmp/ipykernel_897/3453717395.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["current_station", "next_station"])["target_next_arr_delay"]


Historical segment records: 22137025
  current_station next_station  segment_mean_delay  segment_median_delay  \
0            AADR         AADR                 NaN                   NaN   
1            AADR          AAG                 NaN                   NaN   
2            AADR          AAL                 NaN                   NaN   
3            AADR          AAM                 NaN                   NaN   
4            AADR          AAR                 NaN                   NaN   
5            AADR          AAS                 NaN                   NaN   
6            AADR          AAY                 NaN                   NaN   
7            AADR           AB                 NaN                   NaN   
8            AADR          ABD                 NaN                   NaN   
9            AADR          ABI                 NaN                   NaN   

   segment_std_delay  segment_count  
0                NaN              0  
1                NaN              0  
2           

In [108]:
segment_history = (
    train_ns
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )["target_next_arr_delay"]
    .agg(
        segment_mean_delay="mean",
        segment_median_delay="median",
        segment_std_delay="std",
        segment_count="count"
    )
    .reset_index()
)

print("Historical segment records:", len(segment_history))
print(segment_history.head(10))


Historical segment records: 17259
  current_station next_station  segment_mean_delay  segment_median_delay  \
0            AADR         DLPC            4.612903                   0.0   
1            AADR          UHL            7.511364                   4.5   
2             AAG         MKPT           12.937500                   3.0   
3             AAL          APR           77.070175                  24.0   
4             AAL          BUH           75.966387                  22.0   
5             AAL          SDL            9.000000                   9.0   
6             AAM          CQA            0.000000                   0.0   
7             AAM          PKQ            0.000000                   0.0   
8             AAR         BUBR           18.437500                  23.0   
9             AAR          LAA            7.933333                   5.0   

   segment_std_delay  segment_count  
0          20.142407             62  
1          11.583772             88  
2          17.8

In [109]:
print(
    segment_history["segment_count"].describe()
)

print(
    "\nSegments with only 1 observation:",
    (segment_history["segment_count"] == 1).sum()
)

count    17259.000000
mean        54.346022
std         73.499600
min          1.000000
25%         19.000000
50%         23.000000
75%         62.000000
max        908.000000
Name: segment_count, dtype: float64

Segments with only 1 observation: 199


In [110]:
print(
    "Segments with fewer than 5 observations:",
    (segment_history["segment_count"] < 5).sum()
)

print(
    "Segments with fewer than 10 observations:",
    (segment_history["segment_count"] < 10).sum()
)

Segments with fewer than 5 observations: 1046
Segments with fewer than 10 observations: 2155


In [111]:
segment_history["reliable_segment_mean"] = (
    segment_history["segment_mean_delay"]
    .where(segment_history["segment_count"] >= 10)
)

print(
    "Reliable segments:",
    segment_history["reliable_segment_mean"].notna().sum()
)

print(
    "Unreliable segments:",
    segment_history["reliable_segment_mean"].isna().sum()
)

Reliable segments: 15104
Unreliable segments: 2155


In [112]:
# Add historical segment information
train_ns = train_ns.merge(
    segment_history[
        [
            "current_station",
            "next_station",
            "reliable_segment_mean"
        ]
    ],
    on=["current_station", "next_station"],
    how="left"
)

val_ns = val_ns.merge(
    segment_history[
        [
            "current_station",
            "next_station",
            "reliable_segment_mean"
        ]
    ],
    on=["current_station", "next_station"],
    how="left"
)

print("Train shape:", train_ns.shape)
print("Validation shape:", val_ns.shape)

print("\nMissing historical means:")
print("Train:", train_ns["reliable_segment_mean"].isna().sum())
print("Validation:", val_ns["reliable_segment_mean"].isna().sum())

Train shape: (937958, 21)
Validation shape: (286882, 21)

Missing historical means:
Train: 10690
Validation: 3926


In [113]:
fallback_delay = train_ns["target_next_arr_delay"].median()

train_ns["reliable_segment_mean"] = (
    train_ns["reliable_segment_mean"].fillna(fallback_delay)
)

val_ns["reliable_segment_mean"] = (
    val_ns["reliable_segment_mean"].fillna(fallback_delay)
)

print("Fallback delay:", fallback_delay)

print(
    "Missing after fallback:",
    train_ns["reliable_segment_mean"].isna().sum(),
    val_ns["reliable_segment_mean"].isna().sum()
)

Fallback delay: 9.0
Missing after fallback: 0 0


In [114]:
improved_features = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "reliable_segment_mean"
]

X_train_imp = train_ns[improved_features]
y_train_imp = train_ns["target_next_arr_delay"]

X_val_imp = val_ns[improved_features]
y_val_imp = val_ns["target_next_arr_delay"]

categorical_imp = [
    "train",
    "current_station",
    "next_station"
]

print("X_train:", X_train_imp.shape)
print("X_val:", X_val_imp.shape)

X_train: (937958, 9)
X_val: (286882, 9)


In [115]:
categorical_imp = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_imp:
    X_val_imp[col] = X_val_imp[col].astype(
        X_train_imp[col].dtype
    )

    X_val_imp[col] = X_val_imp[col].cat.set_categories(
        X_train_imp[col].cat.categories
    )

/tmp/ipykernel_897/1369875197.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val_imp[col] = X_val_imp[col].astype(
/tmp/ipykernel_897/1369875197.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val_imp[col] = X_val_imp[col].cat.set_categories(
/tmp/ipykernel_897/1369875197.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas

In [116]:
# Make sure date is datetime
train_ns["date"] = pd.to_datetime(train_ns["date"])
val_ns["date"] = pd.to_datetime(val_ns["date"])

# Sort chronologically
train_ns = train_ns.sort_values("date").copy()
val_ns = val_ns.sort_values("date").copy()

print("Training dates:",
      train_ns["date"].min(),
      "to",
      train_ns["date"].max())

print("Validation dates:",
      val_ns["date"].min(),
      "to",
      val_ns["date"].max())

Training dates: 2024-09-01 00:00:00 to 2024-09-23 00:00:00
Validation dates: 2024-09-24 00:00:00 to 2024-09-30 00:00:00


In [117]:
# Calculate daily segment statistics first
daily_segment = (
    train_ns
    .groupby(
        ["date", "current_station", "next_station"],
        observed=True
    )["target_next_arr_delay"]
    .agg(
        daily_segment_mean="mean",
        daily_segment_count="count"
    )
    .reset_index()
)

print("Daily segment records:", len(daily_segment))
print(daily_segment.head())

Daily segment records: 328677
        date current_station next_station  daily_segment_mean  \
0 2024-09-01            AADR         DLPC                 0.0   
1 2024-09-01            AADR          UHL                19.5   
2 2024-09-01             AAG         MKPT                 0.0   
3 2024-09-01             AAL          APR               125.0   
4 2024-09-01             AAL          BUH                68.0   

   daily_segment_count  
0                    2  
1                    4  
2                    1  
3                    3  
4                    3  


In [118]:
# Sort by segment and date
daily_segment = daily_segment.sort_values(
    ["current_station", "next_station", "date"]
).copy()

# Calculate cumulative historical delay
daily_segment["past_segment_mean"] = (
    daily_segment
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )["daily_segment_mean"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

# Calculate how many previous observations we have
daily_segment["past_segment_count"] = (
    daily_segment
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )["daily_segment_count"]
    .transform(
        lambda x: x.shift(1).cumsum()
    )
)

print(
    daily_segment[
        [
            "date",
            "current_station",
            "next_station",
            "daily_segment_mean",
            "past_segment_mean",
            "past_segment_count"
        ]
    ].head(20)
)

             date current_station next_station  daily_segment_mean  \
0      2024-09-01            AADR         DLPC            0.000000   
13750  2024-09-02            AADR         DLPC            0.000000   
27735  2024-09-03            AADR         DLPC            3.333333   
41787  2024-09-04            AADR         DLPC            0.000000   
55932  2024-09-05            AADR         DLPC            8.500000   
70000  2024-09-06            AADR         DLPC            0.000000   
84229  2024-09-07            AADR         DLPC            5.500000   
98172  2024-09-08            AADR         DLPC            0.000000   
112455 2024-09-09            AADR         DLPC            2.333333   
126839 2024-09-10            AADR         DLPC            0.000000   
141055 2024-09-11            AADR         DLPC            0.000000   
155289 2024-09-12            AADR         DLPC            2.666667   
169452 2024-09-13            AADR         DLPC            0.000000   
183790 2024-09-14   

In [119]:
train_ns = train_ns.merge(
    daily_segment[
        [
            "date",
            "current_station",
            "next_station",
            "past_segment_mean",
            "past_segment_count"
        ]
    ],
    on=["date", "current_station", "next_station"],
    how="left"
)

print("Train shape:", train_ns.shape)

print(
    train_ns[
        [
            "date",
            "current_station",
            "next_station",
            "past_segment_mean",
            "past_segment_count"
        ]
    ].head(10)
)

Train shape: (937958, 23)
        date current_station next_station  past_segment_mean  \
0 2024-09-01             CNB           ON                NaN   
1 2024-09-01             UCR          PQN                NaN   
2 2024-09-01             PQN         GRMR                NaN   
3 2024-09-01            GRMR         KHNM                NaN   
4 2024-09-01            KHNM          LGO                NaN   
5 2024-09-01             LGO          PFM                NaN   
6 2024-09-01             PFM          PRG                NaN   
7 2024-09-01             PRG         PYGS                NaN   
8 2024-09-01              VM         CUPJ                NaN   
9 2024-09-01             RGL          JSG                NaN   

   past_segment_count  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4                 NaN  
5                 NaN  
6                 NaN  
7                 NaN  
8                 NaN  
9                 NaN  


In [120]:
print(
    "Rows with past segment history:",
    train_ns["past_segment_mean"].notna().sum()
)

print(
    "Rows without past segment history:",
    train_ns["past_segment_mean"].isna().sum()
)

print(
    "Maximum past observations:",
    train_ns["past_segment_count"].max()
)

Rows with past segment history: 895421
Rows without past segment history: 42537
Maximum past observations: 869.0


In [121]:
fallback_delay = train_ns["target_next_arr_delay"].median()

train_ns["past_segment_mean"] = (
    train_ns["past_segment_mean"].fillna(fallback_delay)
)

train_ns["past_segment_count"] = (
    train_ns["past_segment_count"].fillna(0)
)

print("Fallback delay:", fallback_delay)

print(
    "Missing past mean:",
    train_ns["past_segment_mean"].isna().sum()
)

print(
    "Missing past count:",
    train_ns["past_segment_count"].isna().sum()
)


Fallback delay: 9.0
Missing past mean: 0
Missing past count: 0


In [122]:
# Get the latest historical segment information
# available at the end of the training period
latest_segment_history = (
    daily_segment[
        daily_segment["date"] <= train_ns["date"].max()
    ]
    .sort_values("date")
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )
    .tail(1)
    [
        [
            "current_station",
            "next_station",
            "past_segment_mean"
        ]
    ]
)

print("Segments with validation history:",
      len(latest_segment_history))

print(latest_segment_history.head())

Segments with validation history: 17259
      current_station next_station  past_segment_mean
2925              CPR         PPTA                NaN
18443             GRH          CAO                NaN
25794             SMI          GRH                NaN
24647            RHMA          CTC                NaN
13857             ADX          RXL                NaN


In [123]:
fallback_delay = train_ns["target_next_arr_delay"].median()

latest_segment_history["past_segment_mean"] = (
    latest_segment_history["past_segment_mean"]
    .fillna(fallback_delay)
)

print("Missing validation history:",
      latest_segment_history["past_segment_mean"].isna().sum())

Missing validation history: 0


In [124]:
val_ns = val_ns.merge(
    latest_segment_history[
        [
            "current_station",
            "next_station",
            "past_segment_mean"
        ]
    ],
    on=["current_station", "next_station"],
    how="left"
)

print("Validation shape:", val_ns.shape)

print(
    "Missing validation history:",
    val_ns["past_segment_mean"].isna().sum()
)

Validation shape: (286882, 22)
Missing validation history: 168


In [125]:
val_ns["past_segment_mean"] = (
    val_ns["past_segment_mean"].fillna(fallback_delay)
)

print(
    "Missing validation history:",
    val_ns["past_segment_mean"].isna().sum()
)


Missing validation history: 0


In [126]:
latest_segment_history = (
    daily_segment[
        daily_segment["date"] <= train_ns["date"].max()
    ]
    .sort_values("date")
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )
    .tail(1)
    [
        [
            "current_station",
            "next_station",
            "past_segment_mean",
            "past_segment_count"
        ]
    ]
)

# Fill missing history
latest_segment_history["past_segment_mean"] = (
    latest_segment_history["past_segment_mean"]
    .fillna(fallback_delay)
)

latest_segment_history["past_segment_count"] = (
    latest_segment_history["past_segment_count"]
    .fillna(0)
)

print(latest_segment_history.shape)
print(latest_segment_history.isna().sum())

(17259, 4)
current_station       0
next_station          0
past_segment_mean     0
past_segment_count    0
dtype: int64


In [127]:
val_ns = val_ns.drop(
    columns=["past_segment_mean", "past_segment_count"],
    errors="ignore"
)

In [128]:
val_ns = val_ns.merge(
    latest_segment_history[
        [
            "current_station",
            "next_station",
            "past_segment_mean",
            "past_segment_count"
        ]
    ],
    on=["current_station", "next_station"],
    how="left"
)

val_ns["past_segment_mean"] = (
    val_ns["past_segment_mean"].fillna(fallback_delay)
)

val_ns["past_segment_count"] = (
    val_ns["past_segment_count"].fillna(0)
)

print("Validation shape:", val_ns.shape)

print(
    "Missing mean:",
    val_ns["past_segment_mean"].isna().sum()
)

print(
    "Missing count:",
    val_ns["past_segment_count"].isna().sum()
)

Validation shape: (286882, 23)
Missing mean: 0
Missing count: 0


In [129]:
improved_features_past = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "past_segment_count"
]

X_train_past = train_ns[improved_features_past].copy()
y_train_past = train_ns["target_next_arr_delay"]

X_val_past = val_ns[improved_features_past].copy()
y_val_past = val_ns["target_next_arr_delay"]

categorical_past = [
    "train",
    "current_station",
    "next_station"
]

print("X_train:", X_train_past.shape)
print("X_val:", X_val_past.shape)

X_train: (937958, 10)
X_val: (286882, 10)


In [130]:
for col in categorical_past:
    X_train_past[col] = X_train_past[col].astype("category")
    X_val_past[col] = X_val_past[col].astype("category")

    X_val_past[col] = X_val_past[col].cat.set_categories(
        X_train_past[col].cat.categories
    )

In [131]:
model_past = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_past.fit(
    X_train_past,
    y_train_past,
    categorical_feature=categorical_past
)

pred_past = model_past.predict(X_val_past)

mae_past = mean_absolute_error(y_val_past, pred_past)
rmse_past = np.sqrt(
    mean_squared_error(y_val_past, pred_past)
)

print(f"Past-only Validation MAE: {mae_past:.2f} minutes")
print(f"Past-only Validation RMSE: {rmse_past:.2f} minutes")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.100280 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12378
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 9
[LightGBM] [Info] Start training from score 34.667424
Past-only Validation MAE: 9.00 minutes
Past-only Validation RMSE: 28.55 minutes


In [132]:
daily_segment_extra = (
    train_ns
    .groupby(
        ["date", "current_station", "next_station"],
        observed=True
    )["target_next_arr_delay"]
    .agg(
        daily_mean="mean",
        daily_median="median",
        daily_std="std",
        daily_count="count"
    )
    .reset_index()
)

# A segment occurring only once that day has NaN std.
daily_segment_extra["daily_std"] = (
    daily_segment_extra["daily_std"].fillna(0)
)

print("Daily records:", len(daily_segment_extra))
print(daily_segment_extra.head())

Daily records: 328677
        date current_station next_station  daily_mean  daily_median  \
0 2024-09-01            AADR         DLPC         0.0           0.0   
1 2024-09-01            AADR          UHL        19.5           3.0   
2 2024-09-01             AAG         MKPT         0.0           0.0   
3 2024-09-01             AAL          APR       125.0          32.0   
4 2024-09-01             AAL          BUH        68.0          13.0   

    daily_std  daily_count  
0    0.000000            2  
1   35.038075            4  
2    0.000000            1  
3  189.470314            3  
4   97.000000            3  


In [133]:
daily_segment_extra = daily_segment_extra.sort_values(
    ["current_station", "next_station", "date"]
).copy()

# Past-only median
daily_segment_extra["past_segment_median"] = (
    daily_segment_extra
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )["daily_median"]
    .transform(
        lambda x: x.shift(1).expanding().median()
    )
)

# Past-only standard deviation
daily_segment_extra["past_segment_std"] = (
    daily_segment_extra
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )["daily_std"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

# Past observation count
daily_segment_extra["past_segment_count"] = (
    daily_segment_extra
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )["daily_count"]
    .transform(
        lambda x: x.shift(1).cumsum()
    )
)

print(
    daily_segment_extra[
        [
            "date",
            "current_station",
            "next_station",
            "past_segment_median",
            "past_segment_std",
            "past_segment_count"
        ]
    ].head(20)
)

             date current_station next_station  past_segment_median  \
0      2024-09-01            AADR         DLPC                  NaN   
13750  2024-09-02            AADR         DLPC                  0.0   
27735  2024-09-03            AADR         DLPC                  0.0   
41787  2024-09-04            AADR         DLPC                  0.0   
55932  2024-09-05            AADR         DLPC                  0.0   
70000  2024-09-06            AADR         DLPC                  0.0   
84229  2024-09-07            AADR         DLPC                  0.0   
98172  2024-09-08            AADR         DLPC                  0.0   
112455 2024-09-09            AADR         DLPC                  0.0   
126839 2024-09-10            AADR         DLPC                  0.0   
141055 2024-09-11            AADR         DLPC                  0.0   
155289 2024-09-12            AADR         DLPC                  0.0   
169452 2024-09-13            AADR         DLPC                  0.0   
183790

In [134]:
train_ns = train_ns.drop(
    columns=[
        "past_segment_median",
        "past_segment_std",
        "past_segment_count"
    ],
    errors="ignore"
)

train_ns = train_ns.merge(
    daily_segment_extra[
        [
            "date",
            "current_station",
            "next_station",
            "past_segment_median",
            "past_segment_std",
            "past_segment_count"
        ]
    ],
    on=["date", "current_station", "next_station"],
    how="left"
)

print("Train shape:", train_ns.shape)

print(
    train_ns[
        [
            "past_segment_mean",
            "past_segment_median",
            "past_segment_std",
            "past_segment_count"
        ]
    ].head(10)
)

Train shape: (937958, 25)
   past_segment_mean  past_segment_median  past_segment_std  \
0                9.0                  NaN               NaN   
1                9.0                  NaN               NaN   
2                9.0                  NaN               NaN   
3                9.0                  NaN               NaN   
4                9.0                  NaN               NaN   
5                9.0                  NaN               NaN   
6                9.0                  NaN               NaN   
7                9.0                  NaN               NaN   
8                9.0                  NaN               NaN   
9                9.0                  NaN               NaN   

   past_segment_count  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4                 NaN  
5                 NaN  
6                 NaN  
7                 NaN  
8                 NaN  
9                 NaN  


In [135]:
fallback_delay = train_ns["target_next_arr_delay"].median()

train_ns["past_segment_median"] = (
    train_ns["past_segment_median"].fillna(fallback_delay)
)

train_ns["past_segment_std"] = (
    train_ns["past_segment_std"].fillna(0)
)

train_ns["past_segment_count"] = (
    train_ns["past_segment_count"].fillna(0)
)

print("Missing median:",
      train_ns["past_segment_median"].isna().sum())

print("Missing std:",
      train_ns["past_segment_std"].isna().sum())

print("Missing count:",
      train_ns["past_segment_count"].isna().sum())

Missing median: 0
Missing std: 0
Missing count: 0


In [136]:
# Remove old versions if they exist
val_ns = val_ns.drop(
    columns=[
        "past_segment_median",
        "past_segment_std",
        "past_segment_count"
    ],
    errors="ignore"
)

# Get the latest historical values available before validation
latest_extra_history = (
    daily_segment_extra[
        daily_segment_extra["date"] <= train_ns["date"].max()
    ]
    .sort_values("date")
    .groupby(
        ["current_station", "next_station"],
        observed=True
    )
    .tail(1)
    [
        [
            "current_station",
            "next_station",
            "past_segment_median",
            "past_segment_std",
            "past_segment_count"
        ]
    ]
)

# Merge into validation
val_ns = val_ns.merge(
    latest_extra_history,
    on=["current_station", "next_station"],
    how="left"
)

# Fill unseen segments
val_ns["past_segment_median"] = (
    val_ns["past_segment_median"].fillna(fallback_delay)
)

val_ns["past_segment_std"] = (
    val_ns["past_segment_std"].fillna(0)
)

val_ns["past_segment_count"] = (
    val_ns["past_segment_count"].fillna(0)
)

print("Validation shape:", val_ns.shape)

print("Missing median:",
      val_ns["past_segment_median"].isna().sum())

print("Missing std:",
      val_ns["past_segment_std"].isna().sum())

print("Missing count:",
      val_ns["past_segment_count"].isna().sum())

Validation shape: (286882, 25)
Missing median: 0
Missing std: 0
Missing count: 0


In [137]:
features_final = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "past_segment_median",
    "past_segment_std",
    "past_segment_count"
]

X_train_final = train_ns[features_final].copy()
y_train_final = train_ns["target_next_arr_delay"]

X_val_final = val_ns[features_final].copy()
y_val_final = val_ns["target_next_arr_delay"]

categorical_final = [
    "train",
    "current_station",
    "next_station"
]

print("X_train:", X_train_final.shape)
print("X_val:", X_val_final.shape)

X_train: (937958, 12)
X_val: (286882, 12)


In [138]:
for col in categorical_final:
    X_train_final[col] = X_train_final[col].astype("category")
    X_val_final[col] = X_val_final[col].astype("category")

    X_val_final[col] = X_val_final[col].cat.set_categories(
        X_train_final[col].cat.categories
    )

In [139]:
model_final = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_final.fit(
    X_train_final,
    y_train_final,
    categorical_feature=categorical_final
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.130397 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12888
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 11
[LightGBM] [Info] Start training from score 34.667424


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [140]:
pred_final = model_final.predict(X_val_final)

mae_final = mean_absolute_error(y_val_final, pred_final)
rmse_final = np.sqrt(
    mean_squared_error(y_val_final, pred_final)
)

print(f"Final Validation MAE: {mae_final:.2f} minutes")
print(f"Final Validation RMSE: {rmse_final:.2f} minutes")

Final Validation MAE: 9.01 minutes
Final Validation RMSE: 28.54 minutes


In [141]:
daily_train = (
    train_ns
    .groupby(
        ["date", "train"],
        observed=True
    )["target_next_arr_delay"]
    .agg(
        daily_train_mean="mean",
        daily_train_count="count"
    )
    .reset_index()
)

print("Daily train records:", len(daily_train))
print(daily_train.head(10))

Daily train records: 43975
        date train  daily_train_mean  daily_train_count
0 2024-09-01   962          9.000000                  2
1 2024-09-01  1023         48.714286                 21
2 2024-09-01  1026          1.000000                 24
3 2024-09-01  1027         49.807692                 26
4 2024-09-01  1140         13.666667                 24
5 2024-09-01  1155         15.148148                 27
6 2024-09-01  1156         40.703704                 27
7 2024-09-01  1167        112.450000                 20
8 2024-09-01  1171         22.818182                 22
9 2024-09-01  1172         14.863636                 22


In [142]:
daily_train = daily_train.sort_values(
    ["train", "date"]
).copy()

daily_train["past_train_mean"] = (
    daily_train
    .groupby("train", observed=True)["daily_train_mean"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

daily_train["past_train_count"] = (
    daily_train
    .groupby("train", observed=True)["daily_train_count"]
    .transform(
        lambda x: x.shift(1).cumsum()
    )
)

print(
    daily_train[
        [
            "date",
            "train",
            "daily_train_mean",
            "past_train_mean",
            "past_train_count"
        ]
    ].head(20)
)

            date train  daily_train_mean  past_train_mean  past_train_count
7316  2024-09-05   961          0.000000              NaN               NaN
18629 2024-09-11   961         46.500000         0.000000               2.0
20503 2024-09-12   961          0.000000        23.250000               4.0
24346 2024-09-14   961          9.000000        15.500000               6.0
26269 2024-09-15   961          0.000000        13.875000               8.0
40080 2024-09-22   961         22.000000        11.100000              10.0
0     2024-09-01   962          9.000000              NaN               NaN
1803  2024-09-02   962         14.000000         9.000000               2.0
20504 2024-09-12   962         11.000000        11.500000               4.0
24347 2024-09-14   962         18.000000        11.333333               6.0
26270 2024-09-15   962          8.500000        13.000000               8.0
32195 2024-09-18   962          5.500000        12.100000              10.0
34190 2024-0

In [143]:
train_ns = train_ns.drop(
    columns=["past_train_mean", "past_train_count"],
    errors="ignore"
)

train_ns = train_ns.merge(
    daily_train[
        [
            "date",
            "train",
            "past_train_mean",
            "past_train_count"
        ]
    ],
    on=["date", "train"],
    how="left"
)

print("Train shape:", train_ns.shape)

print(
    train_ns[
        [
            "train",
            "date",
            "past_train_mean",
            "past_train_count"
        ]
    ].head(10)
)

Train shape: (937958, 27)
   train       date  past_train_mean  past_train_count
0   4137 2024-09-01              NaN               NaN
1  14218 2024-09-01              NaN               NaN
2  14218 2024-09-01              NaN               NaN
3  14218 2024-09-01              NaN               NaN
4  14218 2024-09-01              NaN               NaN
5  14218 2024-09-01              NaN               NaN
6  14218 2024-09-01              NaN               NaN
7  14218 2024-09-01              NaN               NaN
8  16368 2024-09-01              NaN               NaN
9  18006 2024-09-01              NaN               NaN


In [144]:
fallback_delay = 9.0

train_ns["past_train_mean"] = (
    train_ns["past_train_mean"].fillna(fallback_delay)
)

train_ns["past_train_count"] = (
    train_ns["past_train_count"].fillna(0)
)

print(
    "Missing past train mean:",
    train_ns["past_train_mean"].isna().sum()
)

print(
    "Missing past train count:",
    train_ns["past_train_count"].isna().sum()
)

Missing past train mean: 0
Missing past train count: 0


In [145]:
# Remove old versions if they exist
val_ns = val_ns.drop(
    columns=["past_train_mean", "past_train_count"],
    errors="ignore"
)

# Only history available before validation starts
latest_train_history = (
    daily_train[
        daily_train["date"] <= train_ns["date"].max()
    ]
    .sort_values("date")
    .groupby("train", observed=True)
    .tail(1)
    [
        [
            "train",
            "past_train_mean",
            "past_train_count"
        ]
    ]
)

# Merge historical train behavior into validation
val_ns = val_ns.merge(
    latest_train_history,
    on="train",
    how="left"
)

# Fallback for trains with no historical record
val_ns["past_train_mean"] = (
    val_ns["past_train_mean"].fillna(fallback_delay)
)

val_ns["past_train_count"] = (
    val_ns["past_train_count"].fillna(0)
)

print("Validation shape:", val_ns.shape)

print(
    "Missing validation train mean:",
    val_ns["past_train_mean"].isna().sum()
)

print(
    "Missing validation train count:",
    val_ns["past_train_count"].isna().sum()
)

Validation shape: (286882, 27)
Missing validation train mean: 0
Missing validation train count: 0


In [146]:
features_train_history = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "past_segment_median",
    "past_segment_std",
    "past_segment_count",
    "past_train_mean",
    "past_train_count"
]

X_train_th = train_ns[features_train_history].copy()
y_train_th = train_ns["target_next_arr_delay"]

X_val_th = val_ns[features_train_history].copy()
y_val_th = val_ns["target_next_arr_delay"]

categorical_th = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_th:
    X_train_th[col] = X_train_th[col].astype("category")
    X_val_th[col] = X_val_th[col].astype("category")

    X_val_th[col] = X_val_th[col].cat.set_categories(
        X_train_th[col].cat.categories
    )

print("X_train:", X_train_th.shape)
print("X_val:", X_val_th.shape)

X_train: (937958, 14)
X_val: (286882, 14)


In [147]:
model_train_history = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_train_history.fit(
    X_train_th,
    y_train_th,
    categorical_feature=categorical_th
)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.117576 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13398
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 13
[LightGBM] [Info] Start training from score 34.667424


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [148]:
pred_th = model_train_history.predict(X_val_th)

mae_th = mean_absolute_error(y_val_th, pred_th)
rmse_th = np.sqrt(
    mean_squared_error(y_val_th, pred_th)
)

print(f"Train-history Validation MAE: {mae_th:.2f} minutes")
print(f"Train-history Validation RMSE: {rmse_th:.2f} minutes")

Train-history Validation MAE: 9.11 minutes
Train-history Validation RMSE: 29.01 minutes


In [149]:
train_ns["delay_excess"] = (
    train_ns["current_arr_delay"]
    - train_ns["past_segment_mean"]
)

val_ns["delay_excess"] = (
    val_ns["current_arr_delay"]
    - val_ns["past_segment_mean"]
)

print(
    train_ns[
        [
            "current_arr_delay",
            "past_segment_mean",
            "delay_excess"
        ]
    ].head(10)
)

   current_arr_delay  past_segment_mean  delay_excess
0               10.0                9.0           1.0
1               55.0                9.0          46.0
2               54.0                9.0          45.0
3               55.0                9.0          46.0
4               67.0                9.0          58.0
5               74.0                9.0          65.0
6               56.0                9.0          47.0
7               54.0                9.0          45.0
8              277.0                9.0         268.0
9                0.0                9.0          -9.0


In [150]:
features_excess = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "delay_excess"
]

X_train_excess = train_ns[features_excess].copy()
y_train_excess = train_ns["target_next_arr_delay"]

X_val_excess = val_ns[features_excess].copy()
y_val_excess = val_ns["target_next_arr_delay"]

categorical_excess = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_excess:
    X_train_excess[col] = X_train_excess[col].astype("category")
    X_val_excess[col] = X_val_excess[col].astype("category")

    X_val_excess[col] = X_val_excess[col].cat.set_categories(
        X_train_excess[col].cat.categories
    )

print("X_train:", X_train_excess.shape)
print("X_val:", X_val_excess.shape)

X_train: (937958, 10)
X_val: (286882, 10)


In [151]:
model_excess = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_excess.fit(
    X_train_excess,
    y_train_excess,
    categorical_feature=categorical_excess
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.165667 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12378
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 9
[LightGBM] [Info] Start training from score 34.667424


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [152]:
pred_excess = model_excess.predict(X_val_excess)

mae_excess = mean_absolute_error(
    y_val_excess,
    pred_excess
)

rmse_excess = np.sqrt(
    mean_squared_error(y_val_excess, pred_excess)
)

print(f"Delay-excess Validation MAE: {mae_excess:.2f} minutes")
print(f"Delay-excess Validation RMSE: {rmse_excess:.2f} minutes")

Delay-excess Validation MAE: 9.01 minutes
Delay-excess Validation RMSE: 28.34 minutes


In [153]:
train_ns["target_delay_change"] = (
    train_ns["target_next_arr_delay"]
    - train_ns["current_arr_delay"]
)

val_ns["target_delay_change"] = (
    val_ns["target_next_arr_delay"]
    - val_ns["current_arr_delay"]
)

print(train_ns["target_delay_change"].describe())
print()
print(val_ns["target_delay_change"].describe())

count    937958.000000
mean          0.692514
std          34.080560
min       -4333.000000
25%          -2.000000
50%           0.000000
75%           4.000000
max        4322.000000
Name: target_delay_change, dtype: float64

count    286882.000000
mean          0.700797
std          31.486868
min       -2037.000000
25%          -2.000000
50%           0.000000
75%           4.000000
max        1888.000000
Name: target_delay_change, dtype: float64


In [156]:
model_change = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_change.fit(
    X_train_excess,
    train_ns["target_delay_change"],
    categorical_feature=categorical_excess
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.101470 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12378
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 9
[LightGBM] [Info] Start training from score 0.692514


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [157]:
pred_change = model_change.predict(X_val_excess)

pred_next_change = (
    val_ns["current_arr_delay"].values
    + pred_change
)
pred_next_change = np.maximum(pred_next_change, 0)

In [158]:
mae_change = mean_absolute_error(
    val_ns["target_next_arr_delay"],
    pred_next_change
)

rmse_change = np.sqrt(
    mean_squared_error(
        val_ns["target_next_arr_delay"],
        pred_next_change
    )
)

print(f"Delay-change Validation MAE: {mae_change:.2f} minutes")
print(f"Delay-change Validation RMSE: {rmse_change:.2f} minutes")

Delay-change Validation MAE: 8.80 minutes
Delay-change Validation RMSE: 28.52 minutes


In [159]:
error_change = val_ns.copy()

error_change["prediction"] = pred_next_change

error_change["absolute_error"] = (
    error_change["target_next_arr_delay"]
    - error_change["prediction"]
).abs()

error_change["delay_group"] = pd.cut(
    error_change["current_arr_delay"],
    bins=[-1, 0, 10, 30, 60, 120, float("inf")],
    labels=[
        "0 min",
        "1-10 min",
        "11-30 min",
        "31-60 min",
        "61-120 min",
        "120+ min"
    ]
)

print(
    error_change
    .groupby("delay_group", observed=True)["absolute_error"]
    .agg(["mean", "median", "count"])
)

                  mean    median  count
delay_group                            
0 min         8.080024  3.231985  97322
1-10 min      5.216242  3.441473  56272
11-30 min     6.429370  3.870607  65540
31-60 min     9.187089  5.054946  32754
61-120 min   13.557277  6.566765  18542
120+ min     28.700054  9.743803  16452


In [160]:
zero_delay = val_ns[
    val_ns["current_arr_delay"] == 0
].copy()

zero_delay["absolute_error"] = (
    zero_delay["target_next_arr_delay"]
    - zero_delay["current_arr_delay"]
).abs()

print(
    zero_delay.groupby(
        pd.qcut(
            zero_delay["past_segment_mean"],
            q=5,
            duplicates="drop"
        ),
        observed=True
    )["target_next_arr_delay"]
    .agg(["mean", "median", "count"])
)

                        mean  median  count
past_segment_mean                          
(-0.001, 8.314]     2.104931     0.0  19470
(8.314, 16.364]     4.672523     0.0  19467
(16.364, 27.363]    6.044356     0.0  19456
(27.363, 47.824]    7.264074     0.0  19468
(47.824, 712.75]   11.778634     0.0  19461


In [161]:
train_ns["segment_risk_adjusted"] = (
    train_ns["past_segment_mean"]
    * (1 + train_ns["current_arr_delay"] / 60)
)

val_ns["segment_risk_adjusted"] = (
    val_ns["past_segment_mean"]
    * (1 + val_ns["current_arr_delay"] / 60)
)

print(
    train_ns[
        [
            "current_arr_delay",
            "past_segment_mean",
            "segment_risk_adjusted"
        ]
    ].head(10)
)

   current_arr_delay  past_segment_mean  segment_risk_adjusted
0               10.0                9.0                  10.50
1               55.0                9.0                  17.25
2               54.0                9.0                  17.10
3               55.0                9.0                  17.25
4               67.0                9.0                  19.05
5               74.0                9.0                  20.10
6               56.0                9.0                  17.40
7               54.0                9.0                  17.10
8              277.0                9.0                  50.55
9                0.0                9.0                   9.00


In [162]:
features_risk = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "delay_excess",
    "segment_risk_adjusted"
]

X_train_risk = train_ns[features_risk].copy()
X_val_risk = val_ns[features_risk].copy()

y_train_risk = train_ns["target_delay_change"]
y_val_risk = val_ns["target_delay_change"]

categorical_risk = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_risk:
    X_train_risk[col] = X_train_risk[col].astype("category")
    X_val_risk[col] = X_val_risk[col].astype("category")

    X_val_risk[col] = X_val_risk[col].cat.set_categories(
        X_train_risk[col].cat.categories
    )

print("X_train:", X_train_risk.shape)
print("X_val:", X_val_risk.shape)

X_train: (937958, 11)
X_val: (286882, 11)


In [163]:
model_risk = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_risk.fit(
    X_train_risk,
    y_train_risk,
    categorical_feature=categorical_risk
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.111801 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12633
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 10
[LightGBM] [Info] Start training from score 0.692514


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [164]:
pred_risk_change = model_risk.predict(X_val_risk)

# Convert predicted change → predicted next-station delay
pred_risk = (
    val_ns["current_arr_delay"].values
    + pred_risk_change
)

# Delay can't be negative
pred_risk = np.maximum(pred_risk, 0)

In [165]:
mae_risk = mean_absolute_error(
    val_ns["target_next_arr_delay"],
    pred_risk
)

rmse_risk = np.sqrt(
    mean_squared_error(
        val_ns["target_next_arr_delay"],
        pred_risk
    )
)

print(f"Risk-adjusted Validation MAE: {mae_risk:.2f} minutes")
print(f"Risk-adjusted Validation RMSE: {rmse_risk:.2f} minutes")

Risk-adjusted Validation MAE: 8.79 minutes
Risk-adjusted Validation RMSE: 28.46 minutes


In [166]:
importance_risk = pd.DataFrame({
    "feature": X_train_risk.columns,
    "importance": model_risk.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance_risk)

                      feature  importance
0                       train        3374
2                next_station        2592
1             current_station        2477
3           current_arr_delay        2380
9                delay_excess        1333
8           past_segment_mean         928
4   scheduled_segment_minutes         852
10      segment_risk_adjusted         664
6                 day_of_week         400
5                       month           0
7                  is_weekend           0


In [167]:
features_no_risk = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "delay_excess"
]

X_train_no_risk = train_ns[features_no_risk].copy()
X_val_no_risk = val_ns[features_no_risk].copy()

y_train_no_risk = train_ns["target_delay_change"]
y_val_no_risk = val_ns["target_delay_change"]

categorical_no_risk = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_no_risk:
    X_train_no_risk[col] = X_train_no_risk[col].astype("category")
    X_val_no_risk[col] = X_val_no_risk[col].astype("category")

    X_val_no_risk[col] = X_val_no_risk[col].cat.set_categories(
        X_train_no_risk[col].cat.categories
    )

print("X_train:", X_train_no_risk.shape)
print("X_val:", X_val_no_risk.shape)

X_train: (937958, 10)
X_val: (286882, 10)


In [168]:
model_no_risk = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_no_risk.fit(
    X_train_no_risk,
    y_train_no_risk,
    categorical_feature=categorical_no_risk
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.098491 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12378
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 9
[LightGBM] [Info] Start training from score 0.692514


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [169]:
pred_no_risk_change = model_no_risk.predict(X_val_no_risk)

pred_no_risk = (
    val_ns["current_arr_delay"].values
    + pred_no_risk_change
)

pred_no_risk = np.maximum(pred_no_risk, 0)

mae_no_risk = mean_absolute_error(
    val_ns["target_next_arr_delay"],
    pred_no_risk
)

rmse_no_risk = np.sqrt(
    mean_squared_error(
        val_ns["target_next_arr_delay"],
        pred_no_risk
    )
)

print(f"No-risk Validation MAE: {mae_no_risk:.2f} minutes")
print(f"No-risk Validation RMSE: {rmse_no_risk:.2f} minutes")

No-risk Validation MAE: 8.80 minutes
No-risk Validation RMSE: 28.52 minutes


In [170]:
features_no_excess = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean"
]

X_train_no_excess = train_ns[features_no_excess].copy()
X_val_no_excess = val_ns[features_no_excess].copy()

y_train_no_excess = train_ns["target_delay_change"]
y_val_no_excess = val_ns["target_delay_change"]

categorical_no_excess = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_no_excess:
    X_train_no_excess[col] = X_train_no_excess[col].astype("category")
    X_val_no_excess[col] = X_val_no_excess[col].astype("category")

    X_val_no_excess[col] = X_val_no_excess[col].cat.set_categories(
        X_train_no_excess[col].cat.categories
    )

print("X_train:", X_train_no_excess.shape)
print("X_val:", X_val_no_excess.shape)

X_train: (937958, 9)
X_val: (286882, 9)


In [171]:
model_no_excess = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_no_excess.fit(
    X_train_no_excess,
    y_train_no_excess,
    categorical_feature=categorical_no_excess
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.103971 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12123
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 8
[LightGBM] [Info] Start training from score 0.692514


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [172]:
pred_no_excess_change = model_no_excess.predict(X_val_no_excess)

pred_no_excess = (
    val_ns["current_arr_delay"].values
    + pred_no_excess_change
)

pred_no_excess = np.maximum(pred_no_excess, 0)

mae_no_excess = mean_absolute_error(
    val_ns["target_next_arr_delay"],
    pred_no_excess
)

rmse_no_excess = np.sqrt(
    mean_squared_error(
        val_ns["target_next_arr_delay"],
        pred_no_excess
    )
)

print(f"No-excess Validation MAE: {mae_no_excess:.2f} minutes")
print(f"No-excess Validation RMSE: {rmse_no_excess:.2f} minutes")

No-excess Validation MAE: 8.78 minutes
No-excess Validation RMSE: 28.48 minutes


In [173]:
features_no_segment = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend"
]

X_train_no_segment = train_ns[features_no_segment].copy()
X_val_no_segment = val_ns[features_no_segment].copy()

y_train_no_segment = train_ns["target_delay_change"]
y_val_no_segment = val_ns["target_delay_change"]

categorical_no_segment = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_no_segment:
    X_train_no_segment[col] = X_train_no_segment[col].astype("category")
    X_val_no_segment[col] = X_val_no_segment[col].astype("category")

    X_val_no_segment[col] = X_val_no_segment[col].cat.set_categories(
        X_train_no_segment[col].cat.categories
    )

print("X_train:", X_train_no_segment.shape)
print("X_val:", X_val_no_segment.shape)


X_train: (937958, 8)
X_val: (286882, 8)


In [174]:
model_no_segment = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_no_segment.fit(
    X_train_no_segment,
    y_train_no_segment,
    categorical_feature=categorical_no_segment
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030294 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11868
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 7
[LightGBM] [Info] Start training from score 0.692514


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [175]:
pred_no_segment_change = model_no_segment.predict(
    X_val_no_segment
)

pred_no_segment = (
    val_ns["current_arr_delay"].values
    + pred_no_segment_change
)

pred_no_segment = np.maximum(pred_no_segment, 0)

mae_no_segment = mean_absolute_error(
    val_ns["target_next_arr_delay"],
    pred_no_segment
)

rmse_no_segment = np.sqrt(
    mean_squared_error(
        val_ns["target_next_arr_delay"],
        pred_no_segment
    )
)

print(f"No-segment Validation MAE: {mae_no_segment:.2f} minutes")
print(f"No-segment Validation RMSE: {rmse_no_segment:.2f} minutes")

No-segment Validation MAE: 8.87 minutes
No-segment Validation RMSE: 28.48 minutes


In [176]:
residuals = (
    val_ns["target_next_arr_delay"].values
    - pred_no_segment
)

print("Mean residual:", residuals.mean())
print("Median residual:", np.median(residuals))

print(
    "Mean prediction:",
    pred_no_segment.mean()
)

print(
    "Mean actual:",
    val_ns["target_next_arr_delay"].mean()
)

Mean residual: -0.15147967734089104
Median residual: -1.3509368886234443
Mean prediction: 31.591207509690086
Mean actual: 31.439727832349188


In [177]:
pred_best_change = model_no_excess.predict(X_val_no_excess)

pred_best = (
    val_ns["current_arr_delay"].values
    + pred_best_change
)

pred_best = np.maximum(pred_best, 0)

In [178]:
residuals = (
    val_ns["target_next_arr_delay"].values
    - pred_best
)

print("Mean residual:", residuals.mean())
print("Median residual:", np.median(residuals))

print("Mean prediction:", pred_best.mean())
print(
    "Mean actual:",
    val_ns["target_next_arr_delay"].mean()
)

Mean residual: -0.23226638540888841
Median residual: -1.337035076952671
Mean prediction: 31.671994217758073
Mean actual: 31.439727832349188


In [179]:
bias_analysis = val_ns.copy()

bias_analysis["prediction"] = pred_best

bias_analysis["residual"] = (
    bias_analysis["target_next_arr_delay"]
    - bias_analysis["prediction"]
)

bias_analysis["delay_group"] = pd.cut(
    bias_analysis["current_arr_delay"],
    bins=[-1, 0, 10, 30, 60, 120, float("inf")],
    labels=[
        "0 min",
        "1-10 min",
        "11-30 min",
        "31-60 min",
        "61-120 min",
        "120+ min"
    ]
)

print(
    bias_analysis
    .groupby("delay_group", observed=True)["residual"]
    .agg(["mean", "median", "count"])
)

                 mean    median  count
delay_group                           
0 min       -0.610485 -2.267137  97322
1-10 min     0.022023 -1.303424  56272
11-30 min   -0.018767 -0.585302  65540
31-60 min   -0.085065  0.000000  32754
61-120 min  -0.203364  0.389963  18542
120+ min    -0.040830  2.781612  16452


In [180]:
error_analysis = val_ns.copy()

error_analysis["prediction"] = pred_best

error_analysis["absolute_error"] = (
    error_analysis["target_next_arr_delay"]
    - error_analysis["prediction"]
).abs()

error_analysis["actual_group"] = pd.cut(
    error_analysis["target_next_arr_delay"],
    bins=[-1, 0, 10, 30, 60, 120, float("inf")],
    labels=[
        "0 min",
        "1-10 min",
        "11-30 min",
        "31-60 min",
        "61-120 min",
        "120+ min"
    ]
)

print(
    error_analysis
    .groupby("actual_group", observed=True)["absolute_error"]
    .agg(["mean", "median", "count"])
)

                   mean     median  count
actual_group                             
0 min          8.425936   3.551867  97279
1-10 min       4.088166   2.645817  54740
11-30 min      5.591050   3.742419  66066
31-60 min      9.523154   5.500169  33065
61-120 min    14.547780   7.114262  18817
120+ min      30.606080  10.541312  16915


In [181]:
worst = val_ns.copy()

worst["prediction"] = pred_best

worst["absolute_error"] = (
    worst["target_next_arr_delay"]
    - worst["prediction"]
).abs()

print(
    worst[
        [
            "train",
            "date",
            "current_station",
            "next_station",
            "current_arr_delay",
            "past_segment_mean",
            "target_next_arr_delay",
            "prediction",
            "absolute_error"
        ]
    ]
    .sort_values("absolute_error", ascending=False)
    .head(20)
)

        train       date current_station next_station  current_arr_delay  \
252891   1028 2024-09-30             KYN           DR             2037.0   
246455  12226 2024-09-30             SMZ          AMH                0.0   
253117   1028 2024-09-30             LAR         BINA                0.0   
226624  12225 2024-09-29             RDL          BBK                0.0   
228682  12226 2024-09-29             SMZ          AMH                0.0   
189649  12225 2024-09-28             GZB          DLI                0.0   
118248  12261 2024-09-26            TATA          HWH                0.0   
189188  12226 2024-09-28             SMZ          AMH                0.0   
253134   1028 2024-09-30             MKP         CKTD             1491.0   
246276  12226 2024-09-30             BNZ          BBK             1495.0   
189596  12225 2024-09-28             RDL          BBK                0.0   
115669   1028 2024-09-26             KYN           DR             1284.0   
153298  1222

In [182]:
train_day_check = (
    train_ns[
        ["train", "date", "current_arr_delay"]
    ]
    .drop_duplicates(["train", "date"])
    .sort_values(["train", "date"])
)

train_day_check["previous_train_delay"] = (
    train_day_check
    .groupby("train")["current_arr_delay"]
    .shift(1)
)

print(
    train_day_check[
        train_day_check["previous_train_delay"].notna()
    ].head(20)
)

       train       date  current_arr_delay  previous_train_delay
408594   961 2024-09-11               58.0                   0.0
454143   961 2024-09-12                0.0                  58.0
562442   961 2024-09-14                4.0                   0.0
603715   961 2024-09-15                0.0                   4.0
861725   961 2024-09-22                4.0                   0.0
70814    962 2024-09-02               10.0                   5.0
454141   962 2024-09-12               22.0                  10.0
562445   962 2024-09-14                2.0                  22.0
603700   962 2024-09-15               13.0                   2.0
705287   962 2024-09-18                0.0                  13.0
745965   962 2024-09-19               10.0                   0.0
854738   962 2024-09-21                5.0                  10.0
70599   1023 2024-09-02               28.0                  79.0
100415  1023 2024-09-03               56.0                  28.0
153749  1023 2024-09-04  

/tmp/ipykernel_897/938859247.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("train")["current_arr_delay"]


In [183]:
train_ns = train_ns.sort_values(["train", "date"]).copy()
val_ns = val_ns.sort_values(["train", "date"]).copy()

train_ns["previous_train_delay"] = (
    train_ns
    .groupby("train", observed=True)["current_arr_delay"]
    .shift(1)
)

val_ns["previous_train_delay"] = (
    val_ns
    .groupby("train", observed=True)["current_arr_delay"]
    .shift(1)
)

In [184]:
print(
    "Train previous history:",
    train_ns["previous_train_delay"].notna().sum()
)

print(
    "Validation previous history:",
    val_ns["previous_train_delay"].notna().sum()
)

Train previous history: 934096
Validation previous history: 283360


In [185]:
# Last known train delay before validation starts
last_train_delay = (
    train_ns
    .sort_values(["train", "date"])
    .groupby("train", observed=True)["current_arr_delay"]
    .last()
)

# Fill missing previous_train_delay in validation
val_ns["previous_train_delay"] = val_ns["previous_train_delay"].fillna(
    val_ns["train"].map(last_train_delay)
)

print(
    "Validation previous history after fix:",
    val_ns["previous_train_delay"].notna().sum()
)

print(
    "Still missing:",
    val_ns["previous_train_delay"].isna().sum()
)

Validation previous history after fix: 286852
Still missing: 30


In [186]:
features_prev_train = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "previous_train_delay"
]

X_train_prev_train = train_ns[features_prev_train].copy()
X_val_prev_train = val_ns[features_prev_train].copy()

y_train_prev_train = train_ns["target_delay_change"]
y_val_prev_train = val_ns["target_delay_change"]

categorical_prev_train = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_prev_train:
    X_train_prev_train[col] = X_train_prev_train[col].astype("category")
    X_val_prev_train[col] = X_val_prev_train[col].astype("category")

    X_val_prev_train[col] = X_val_prev_train[col].cat.set_categories(
        X_train_prev_train[col].cat.categories
    )

print("X_train:", X_train_prev_train.shape)
print("X_val:", X_val_prev_train.shape)


X_train: (937958, 10)
X_val: (286882, 10)


In [187]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [188]:
print("train_ns columns:")
print(train_ns.columns.tolist())

print("\nval_ns columns:")
print(val_ns.columns.tolist())

train_ns columns:
['train', 'date', 'current_station', 'sch_arr', 'act_arr', 'current_arr_delay', 'sch_dep', 'act_dep', 'current_dep_delay', 'next_station', 'target_next_arr_delay', 'next_dep_delay', 'year', 'month', 'day_of_week', 'is_weekend', 'next_sch_arr', 'sch_arr_minutes', 'next_sch_arr_minutes', 'scheduled_segment_minutes', 'reliable_segment_mean', 'past_segment_mean', 'past_segment_median', 'past_segment_std', 'past_segment_count', 'past_train_mean', 'past_train_count', 'delay_excess', 'target_delay_change', 'segment_risk_adjusted', 'previous_train_delay']

val_ns columns:
['train', 'date', 'current_station', 'sch_arr', 'act_arr', 'current_arr_delay', 'sch_dep', 'act_dep', 'current_dep_delay', 'next_station', 'target_next_arr_delay', 'next_dep_delay', 'year', 'month', 'day_of_week', 'is_weekend', 'next_sch_arr', 'sch_arr_minutes', 'next_sch_arr_minutes', 'scheduled_segment_minutes', 'reliable_segment_mean', 'past_segment_mean', 'past_segment_median', 'past_segment_std', 'past_

In [189]:
print("train_ns shape:", train_ns.shape)
print("val_ns shape:", val_ns.shape)

train_ns shape: (937958, 31)
val_ns shape: (286882, 31)


In [190]:
features_prev_train = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "past_segment_mean",
    "day_of_week",
    "month",
    "is_weekend",
    "previous_train_delay"
]

target_prev_train = "target_next_arr_delay"

X_train_prev_train = train_ns[features_prev_train].copy()
y_train_prev_train = train_ns[target_prev_train].copy()

X_val_prev_train = val_ns[features_prev_train].copy()
y_val_prev_train = val_ns[target_prev_train].copy()

# Make station columns categorical
X_train_prev_train["current_station"] = (
    X_train_prev_train["current_station"].astype("category")
)

X_train_prev_train["next_station"] = (
    X_train_prev_train["next_station"].astype("category")
)

# Force validation to use EXACTLY the same categories
X_val_prev_train["current_station"] = (
    pd.Categorical(
        X_val_prev_train["current_station"],
        categories=X_train_prev_train["current_station"].cat.categories
    )
)

X_val_prev_train["next_station"] = (
    pd.Categorical(
        X_val_prev_train["next_station"],
        categories=X_train_prev_train["next_station"].cat.categories
    )
)

categorical_prev_train = [
    "current_station",
    "next_station"
]

print("X_train:", X_train_prev_train.shape)
print("X_val:", X_val_prev_train.shape)
print("Features:", len(features_prev_train))

X_train: (937958, 10)
X_val: (286882, 10)
Features: 10


In [191]:
model_prev_train = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_prev_train.fit(
    X_train_prev_train,
    y_train_prev_train,
    categorical_feature=categorical_prev_train
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.182124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9273
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 9
[LightGBM] [Info] Start training from score 34.667424


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [192]:
# Force validation categorical columns to exactly match training
for col in categorical_prev_train:
    X_val_prev_train[col] = pd.Categorical(
        X_val_prev_train[col],
        categories=X_train_prev_train[col].cat.categories
    )

print("Category check:")

for col in categorical_prev_train:
    train_cats = X_train_prev_train[col].cat.categories
    val_cats = X_val_prev_train[col].cat.categories

    print(
        col,
        "train:", len(train_cats),
        "validation:", len(val_cats),
        "identical:", train_cats.equals(val_cats)
    )

Category check:
current_station train: 4705 validation: 4705 identical: True
next_station train: 4705 validation: 4705 identical: True


In [193]:
print("MODEL STORED CATEGORICAL DATA:")
print(model_prev_train.booster_.pandas_categorical)

print("\nDATAFRAME CATEGORICAL COLUMNS:")
print(X_train_prev_train.select_dtypes(include="category").columns.tolist())
print(X_val_prev_train.select_dtypes(include="category").columns.tolist())

print("\nCATEGORY DETAILS:")

for col in categorical_prev_train:
    train_cats = X_train_prev_train[col].cat.categories
    val_cats = X_val_prev_train[col].cat.categories

    print(f"\n{col}")
    print("Train categories:", len(train_cats))
    print("Val categories:  ", len(val_cats))
    print("Same categories:", train_cats.equals(val_cats))
    print("Same dtype:     ", train_cats.dtype == val_cats.dtype)

MODEL STORED CATEGORICAL DATA:
[[961, 962, 1007, 1008, 1023, 1024, 1025, 1026, 1027, 1028, 1031, 1032, 1091, 1103, 1104, 1139, 1140, 1155, 1156, 1163, 1164, 1165, 1166, 1167, 1168, 1171, 1172, 1181, 1183, 1185, 1186, 1203, 1211, 1212, 1303, 1307, 1315, 1317, 1319, 1321, 1323, 1351, 1353, 1355, 1357, 1359, 1361, 1365, 1367, 1369, 1401, 1435, 1436, 1437, 1445, 1446, 1461, 1462, 1487, 1488, 1581, 1587, 1589, 1591, 1592, 1595, 1596, 1663, 1664, 1665, 1666, 1667, 1668, 1701, 1702, 1707, 1708, 1883, 1904, 1905, 1906, 1919, 1920, 1921, 1922, 2001, 2023, 2024, 2121, 2122, 2131, 2132, 2133, 2134, 2185, 2186, 2187, 2188, 2197, 2198, 2199, 2200, 2249, 2310, 2391, 2392, 2393, 2394, 2397, 2398, 2525, 2526, 2563, 2564, 2569, 2570, 2575, 2576, 2813, 2814, 2831, 2832, 2837, 2838, 2839, 2840, 2847, 2848, 2897, 2898, 3007, 3008, 3043, 3044, 3047, 3048, 3109, 3110, 3132, 3172, 3183, 3190, 3191, 3204, 3206, 3225, 3226, 3229, 3230, 3239, 3240, 3241, 3242, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 325

In [194]:
for col in categorical_prev_train:
    train_cats = X_train_prev_train[col].cat.categories
    val_cats = X_val_prev_train[col].cat.categories

    print(
        col,
        "| train:", len(train_cats),
        "| val:", len(val_cats),
        "| identical:", train_cats.equals(val_cats)
    )

current_station | train: 4705 | val: 4705 | identical: True
next_station | train: 4705 | val: 4705 | identical: True


In [195]:
# Make copies for prediction
X_val_prev_predict = X_val_prev_train.copy()

# Convert categorical columns to the same integer category codes
for col in categorical_prev_train:
    X_val_prev_predict[col] = (
        X_val_prev_predict[col]
        .cat.codes
        .astype("int32")
    )

# Also convert training data the same way, using the training category mapping
X_train_prev_predict = X_train_prev_train.copy()

for col in categorical_prev_train:
    X_train_prev_predict[col] = (
        X_train_prev_predict[col]
        .cat.codes
        .astype("int32")
    )

print("Prediction data prepared.")
print(X_val_prev_predict.shape)

Prediction data prepared.
(286882, 10)


In [196]:
print("Model feature names:")
print(model_prev_train.booster_.feature_name())

print("\nModel categorical metadata:")
print(model_prev_train.booster_.pandas_categorical is not None)

print("\nTraining dtypes:")
print(X_train_prev_train.dtypes)

Model feature names:
['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'past_segment_mean', 'day_of_week', 'month', 'is_weekend', 'previous_train_delay']

Model categorical metadata:
True

Training dtypes:
train                        category
current_station              category
next_station                 category
current_arr_delay             float64
scheduled_segment_minutes       int32
past_segment_mean             float64
day_of_week                     int32
month                           int32
is_weekend                      int64
previous_train_delay          float64
dtype: object


In [197]:
categorical_prev_train = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_prev_train:
    X_val_prev_train[col] = pd.Categorical(
        X_val_prev_train[col],
        categories=X_train_prev_train[col].cat.categories
    )

print("Categorical columns:")
for col in categorical_prev_train:
    print(
        col,
        "| train:", len(X_train_prev_train[col].cat.categories),
        "| val:", len(X_val_prev_train[col].cat.categories),
        "| identical:",
        X_train_prev_train[col].cat.categories.equals(
            X_val_prev_train[col].cat.categories
        )
    )

Categorical columns:
train | train: 3862 | val: 3862 | identical: True
current_station | train: 4705 | val: 4705 | identical: True
next_station | train: 4705 | val: 4705 | identical: True


In [198]:
pred_prev = model_prev_train.predict(X_val_prev_train)

mae_prev = mean_absolute_error(
    y_val_prev_train,
    pred_prev
)

rmse_prev = np.sqrt(
    mean_squared_error(
        y_val_prev_train,
        pred_prev
    )
)

print(f"Previous-train Validation MAE: {mae_prev:.2f} minutes")
print(f"Previous-train Validation RMSE: {rmse_prev:.2f} minutes")

Previous-train Validation MAE: 8.04 minutes
Previous-train Validation RMSE: 25.23 minutes


In [199]:
print("Current previous_train_delay construction:")
print("Train missing:", train_ns["previous_train_delay"].isna().sum())
print("Validation missing:", val_ns["previous_train_delay"].isna().sum())

print("\nFirst 10 validation values:")
print(
    val_ns[
        ["train", "date", "current_arr_delay", "previous_train_delay"]
    ].head(10)
)

Current previous_train_delay construction:
Train missing: 3862
Validation missing: 30

First 10 validation values:
       train       date  current_arr_delay  previous_train_delay
0       1023 2024-09-24                3.0                  43.0
35801   1023 2024-09-24                0.0                   3.0
35802   1023 2024-09-24               13.0                   0.0
35803   1023 2024-09-24               13.0                  13.0
35804   1023 2024-09-24               16.0                  13.0
35805   1023 2024-09-24               18.0                  16.0
35806   1023 2024-09-24               18.0                  18.0
35807   1023 2024-09-24                0.0                  18.0
35808   1023 2024-09-24                0.0                   0.0
35809   1023 2024-09-24                0.0                   0.0


In [200]:
# Combine train + validation chronologically
combined_test = pd.concat(
    [
        train_ns[["train", "date", "current_arr_delay"]].assign(split="train"),
        val_ns[["train", "date", "current_arr_delay"]].assign(split="val")
    ],
    ignore_index=True
)

combined_test = combined_test.sort_values(
    ["train", "date", "split"]
).reset_index(drop=True)

# Previous occurrence of the same train across BOTH periods
combined_test["previous_train_delay_test"] = (
    combined_test
    .groupby("train", observed=True)["current_arr_delay"]
    .shift(1)
)

# Check validation values
val_test = combined_test[combined_test["split"] == "val"]

print(
    "Validation missing:",
    val_test["previous_train_delay_test"].isna().sum()
)

print(
    val_test[
        ["train", "date", "current_arr_delay", "previous_train_delay_test"]
    ].head(10)
)

Validation missing: 30
     train       date  current_arr_delay  previous_train_delay_test
481   1023 2024-09-24                3.0                       43.0
482   1023 2024-09-24                0.0                        3.0
483   1023 2024-09-24               13.0                        0.0
484   1023 2024-09-24               13.0                       13.0
485   1023 2024-09-24               16.0                       13.0
486   1023 2024-09-24               18.0                       16.0
487   1023 2024-09-24               18.0                       18.0
488   1023 2024-09-24                0.0                       18.0
489   1023 2024-09-24                0.0                        0.0
490   1023 2024-09-24                0.0                        0.0


In [201]:
# Compare the two previous-train features
comparison = val_ns[
    ["train", "date", "current_arr_delay", "previous_train_delay"]
].copy()

comparison["previous_train_delay_test"] = val_test[
    "previous_train_delay_test"
].values

print(comparison.head(20))

print("\nMissing values:")
print(
    "Current lag:",
    comparison["previous_train_delay"].isna().sum()
)
print(
    "Cross-boundary lag:",
    comparison["previous_train_delay_test"].isna().sum()
)

       train       date  current_arr_delay  previous_train_delay  \
0       1023 2024-09-24                3.0                  43.0   
35801   1023 2024-09-24                0.0                   3.0   
35802   1023 2024-09-24               13.0                   0.0   
35803   1023 2024-09-24               13.0                  13.0   
35804   1023 2024-09-24               16.0                  13.0   
35805   1023 2024-09-24               18.0                  16.0   
35806   1023 2024-09-24               18.0                  18.0   
35807   1023 2024-09-24                0.0                  18.0   
35808   1023 2024-09-24                0.0                   0.0   
35809   1023 2024-09-24                0.0                   0.0   
35810   1023 2024-09-24                0.0                   0.0   
35811   1023 2024-09-24                0.0                   0.0   
35812   1023 2024-09-24                0.0                   0.0   
35813   1023 2024-09-24                0.0      

In [202]:
print("Model parameters:")
print(model_prev_train.get_params())

Model parameters:
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 31, 'objective': 'regression', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0}


In [203]:
print("y_train_prev_train:")
print(y_train_prev_train.describe())

print("\ny_val_prev_train:")
print(y_val_prev_train.describe())

print("\nTarget name:")
print(y_train_prev_train.name)

y_train_prev_train:
count    937958.000000
mean         34.667424
std          88.112547
min           0.000000
25%           0.000000
50%           9.000000
75%          30.000000
max        4336.000000
Name: target_next_arr_delay, dtype: float64

y_val_prev_train:
count    286882.000000
mean         31.439728
std          76.557477
min           0.000000
25%           0.000000
50%           9.000000
75%          29.000000
max        2037.000000
Name: target_next_arr_delay, dtype: float64

Target name:
target_next_arr_delay


In [204]:
importance_prev = pd.DataFrame({
    "feature": X_train_prev_train.columns,
    "importance": model_prev_train.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance_prev)

                     feature  importance
9       previous_train_delay        3244
3          current_arr_delay        2963
1            current_station        2913
2               next_station        2837
5          past_segment_mean         986
4  scheduled_segment_minutes         944
0                      train         828
6                day_of_week         285
7                      month           0
8                 is_weekend           0


In [205]:
print("Target name:", y_train_prev_train.name)
print("Validation target:", y_val_prev_train.name)

print("\nTraining target:")
print(y_train_prev_train.describe())

print("\nValidation target:")
print(y_val_prev_train.describe())

Target name: target_next_arr_delay
Validation target: target_next_arr_delay

Training target:
count    937958.000000
mean         34.667424
std          88.112547
min           0.000000
25%           0.000000
50%           9.000000
75%          30.000000
max        4336.000000
Name: target_next_arr_delay, dtype: float64

Validation target:
count    286882.000000
mean         31.439728
std          76.557477
min           0.000000
25%           0.000000
50%           9.000000
75%          29.000000
max        2037.000000
Name: target_next_arr_delay, dtype: float64


In [206]:
print("Current Previous-train features:")
print(features_prev_train)

print("\nModel features:")
print(model_prev_train.booster_.feature_name())

Current Previous-train features:
['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'past_segment_mean', 'day_of_week', 'month', 'is_weekend', 'previous_train_delay']

Model features:
['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'past_segment_mean', 'day_of_week', 'month', 'is_weekend', 'previous_train_delay']


In [207]:
train_ns = train_ns.sort_values(
    ["train", "current_station", "next_station", "date"]
).copy()

val_ns = val_ns.sort_values(
    ["train", "current_station", "next_station", "date"]
).copy()

In [208]:
train_ns["previous_train_segment_delay"] = (
    train_ns
    .groupby(
        ["train", "current_station", "next_station"],
        observed=True
    )["current_arr_delay"]
    .shift(1)
)

In [209]:
# Last known delay for each train + segment before validation
last_train_segment_delay = (
    train_ns
    .sort_values(["train", "current_station", "next_station", "date"])
    .groupby(
        ["train", "current_station", "next_station"],
        observed=True
    )["current_arr_delay"]
    .last()
)

# Previous occurrence within validation
val_ns["previous_train_segment_delay"] = (
    val_ns
    .groupby(
        ["train", "current_station", "next_station"],
        observed=True
    )["current_arr_delay"]
    .shift(1)
)

# Build a Series aligned to val_ns's index
train_history_values = (
    val_ns.set_index(
        ["train", "current_station", "next_station"]
    )
    .index
    .map(last_train_segment_delay)
)

train_history_values = pd.Series(
    train_history_values,
    index=val_ns.index
)

# Fill only the missing first-validation occurrences
val_ns["previous_train_segment_delay"] = (
    val_ns["previous_train_segment_delay"]
    .fillna(train_history_values)
)

print(
    "previous_train_segment_delay missing:",
    val_ns["previous_train_segment_delay"].isna().sum()
)

previous_train_segment_delay missing: 441


In [210]:
# Create a lookup Series for the last training value
last_train_segment_delay = (
    train_ns
    .sort_values(["train", "current_station", "next_station", "date"])
    .groupby(
        ["train", "current_station", "next_station"],
        observed=True
    )["current_arr_delay"]
    .last()
)

# Build a MultiIndex for validation rows
val_keys = pd.MultiIndex.from_frame(
    val_ns[["train", "current_station", "next_station"]]
)

# Map training history onto validation rows
training_previous = pd.Series(
    val_keys.map(last_train_segment_delay),
    index=val_ns.index
)

# Fill only missing validation history
val_ns["previous_train_segment_delay"] = (
    val_ns["previous_train_segment_delay"]
    .fillna(training_previous)
)


In [211]:
print(
    "Train history:",
    train_ns["previous_train_segment_delay"].notna().sum()
)

print(
    "Validation history:",
    val_ns["previous_train_segment_delay"].notna().sum()
)

print(
    val_ns[
        [
            "train",
            "date",
            "current_station",
            "next_station",
            "current_arr_delay",
            "previous_train_segment_delay"
        ]
    ]
    .dropna(subset=["previous_train_segment_delay"])
    .head(20)
)

Train history: 857274
Validation history: 286441
        train       date current_station next_station  current_arr_delay  \
35814    1023 2024-09-24             BVQ          SLI                0.0   
79437    1023 2024-09-25             BVQ          SLI                0.0   
119930   1023 2024-09-26             BVQ          SLI               79.0   
153687   1023 2024-09-27             BVQ          SLI              127.0   
168055   1023 2024-09-28             BVQ          SLI               85.0   
208363   1023 2024-09-29             BVQ          SLI                0.0   
253615   1023 2024-09-30             BVQ          SLI                0.0   
35818    1023 2024-09-24             HTK          RKD                0.0   
79378    1023 2024-09-25             HTK          RKD                0.0   
120137   1023 2024-09-26             HTK          RKD               61.0   
151231   1023 2024-09-27             HTK          RKD                0.0   
168280   1023 2024-09-28             HT

In [212]:
features_prev_segment = [
    "train",
    "current_station",
    "next_station",
    "current_arr_delay",
    "scheduled_segment_minutes",
    "month",
    "day_of_week",
    "is_weekend",
    "past_segment_mean",
    "previous_train_delay",
    "previous_train_segment_delay"
]

X_train_prev_segment = train_ns[features_prev_segment].copy()
X_val_prev_segment = val_ns[features_prev_segment].copy()

y_train_prev_segment = train_ns["target_delay_change"]
y_val_prev_segment = val_ns["target_delay_change"]

categorical_prev_segment = [
    "train",
    "current_station",
    "next_station"
]

for col in categorical_prev_segment:
    X_train_prev_segment[col] = X_train_prev_segment[col].astype("category")
    X_val_prev_segment[col] = X_val_prev_segment[col].astype("category")

    X_val_prev_segment[col] = X_val_prev_segment[col].cat.set_categories(
        X_train_prev_segment[col].cat.categories
    )

print("X_train:", X_train_prev_segment.shape)
print("X_val:", X_val_prev_segment.shape)

X_train: (937958, 11)
X_val: (286882, 11)


In [213]:
model_prev_segment = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_prev_segment.fit(
    X_train_prev_segment,
    y_train_prev_segment,
    categorical_feature=categorical_prev_segment
)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.113034 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12653
[LightGBM] [Info] Number of data points in the train set: 937958, number of used features: 10
[LightGBM] [Info] Start training from score 0.692514


LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)

In [214]:
pred_prev_segment_change = model_prev_segment.predict(
    X_val_prev_segment
)

pred_prev_segment = (
    val_ns["current_arr_delay"].values
    + pred_prev_segment_change
)

pred_prev_segment = np.maximum(pred_prev_segment, 0)

mae_prev_segment = mean_absolute_error(
    val_ns["target_next_arr_delay"],
    pred_prev_segment
)

rmse_prev_segment = np.sqrt(
    mean_squared_error(
        val_ns["target_next_arr_delay"],
        pred_prev_segment
    )
)

print(
    f"Previous-train-segment Validation MAE: "
    f"{mae_prev_segment:.2f} minutes"
)

print(
    f"Previous-train-segment Validation RMSE: "
    f"{rmse_prev_segment:.2f} minutes"
)

Previous-train-segment Validation MAE: 7.88 minutes
Previous-train-segment Validation RMSE: 25.01 minutes


In [215]:
print("=== MODEL COMPARISON ===")
print(f"Old champion:           MAE 7.90 | RMSE 24.89")
print(f"Previous-train-segment: MAE {mae_prev_segment:.2f} | RMSE {rmse_prev_segment:.2f}")

=== MODEL COMPARISON ===
Old champion:           MAE 7.90 | RMSE 24.89
Previous-train-segment: MAE 7.88 | RMSE 25.01


In [216]:
import joblib

joblib.dump(
    model_prev_segment,
    "model_prev_segment_champion.pkl"
)

print("Champion model saved!")

Champion model saved!


In [217]:
joblib.dump(
    {
        "features": features_prev_segment,
        "categorical_features": categorical_prev_segment
    },
    "model_prev_segment_metadata.pkl"
)

print("Model metadata saved!")

Model metadata saved!


In [218]:
import os

print(os.listdir("/content"))

['.config', 'model_prev_segment_metadata.pkl', 'Indian-Railway-Network-and-Delays', 'Indian-Railway-Network-and-Delays.zip', 'model_prev_segment_champion.pkl', 'sample_data']


In [219]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [220]:
import os

print(os.listdir("/content/drive/MyDrive"))

['1-085d360d-67c2-4dee-bd1a-a58c30efc5b0 (1).pdf', 'sparsh-maurya-dc5f476a-63ab-4348-a9d3-6e36001e836b-certificate.pdf', 'Document from Sparsh Maurya', 'SparxhCodes.gdoc', 'IOTREPORTTEAM7.docx', 'COMMUNICATION.pdf', 'SPARSH_PRACTICETEST', 'Certificate.pdf', 'Rumination_Monitoring_System_Dairy_Farmers.docx', 'Untitled document (2).gdoc', 'Classroom', 'create a background of a word document....to make....gsheet', 'Untitled document (1).gdoc', 'Untitled document.gdoc', 'DSTL ESE Model Paper 1.gdoc', 'IMG-20260206-WA0013.jpg', 'Colab Notebooks', 'Screenshot 2026-04-07 093747.png', 'Screenshot 2026-04-07 093837.png', 'Screenshot 2026-04-07 093956.png', 'SPARSH_MAURYA_202501100300257.ipynb', 'VID_20260404_201909.mp4', 'VID20260404201714.mp4', 'IMG_4662.MOV', 'IMG_4604.MOV', 'VID20260527054236.mp4', 'IMG_4657.MOV', 'IMG_4606.MOV', 'IMG_4653.MOV', 'IMG_4652.MOV', 'IMG_4603.MOV', 'IMG_4655.MOV', 'IMG_4631.MOV', 'IMG_4658.MOV', 'IMG_4661.MOV', 'VID_20260404_201909_1.mp4', 'IMG_4656.MOV', 'IMG_46

In [221]:
import joblib

joblib.dump(
    model_prev_train,
    "/content/drive/MyDrive/champion_model.pkl"
)

print("Champion model saved!")

Champion model saved!


In [222]:
import os

path = "/content/drive/MyDrive/champion_model.pkl"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path) / (1024 * 1024), "MB")

Exists: True
Size: 3.6770505905151367 MB


In [223]:
import joblib

joblib.dump(
    model_prev_train,
    "/content/drive/MyDrive/RAILRADAR_7_90_CHAMPION.pkl"
)

print("7.90 champion model saved!")

7.90 champion model saved!


In [224]:
import os

path = "/content/drive/MyDrive/RAILRADAR_7_90_CHAMPION.pkl"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path) / (1024 * 1024), "MB")

Exists: True
Size: 3.6770505905151367 MB


In [225]:
print(model_prev_train.feature_name_)

['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'past_segment_mean', 'day_of_week', 'month', 'is_weekend', 'previous_train_delay']


In [226]:
import lightgbm
print(lightgbm.__version__)
print(type(model_prev_train))

4.6.0
<class 'lightgbm.sklearn.LGBMRegressor'>


In [227]:
print(type(model_prev_train))
print(model_prev_train)
print("Booster:", model_prev_train.booster_)

<class 'lightgbm.sklearn.LGBMRegressor'>
LGBMRegressor(learning_rate=0.05, n_estimators=500, n_jobs=-1,
              objective='regression', random_state=42)
Booster: <lightgbm.basic.Booster object at 0x7e767b6f2550>


In [228]:
print(model_prev_train.booster_.num_trees())
print(model_prev_train.booster_.num_feature())

500
10


In [229]:
try:
    model_prev_train.booster_.save_model("/content/champion_test.txt")
    print("SAVE SUCCESS")
except Exception as e:
    print("ERROR:", repr(e))

SAVE SUCCESS


In [230]:
model_prev_train.booster_.save_model(
    "/content/champion_model.txt"
)

print("Saved successfully!")

Saved successfully!


In [231]:
import os

print(os.path.exists("/content/champion_model.txt"))
print(os.path.getsize("/content/champion_model.txt") / (1024 * 1024), "MB")

True
3.608088493347168 MB


In [232]:
import shutil
import os

drive_path = "/content/drive/MyDrive/model/champion_model.txt"

os.makedirs("/content/drive/MyDrive/model", exist_ok=True)

shutil.copy2(
    "/content/champion_model.txt",
    drive_path
)

print("Copied to Drive:", os.path.exists(drive_path))

Copied to Drive: True


In [233]:
print(os.path.getsize(drive_path) / (1024 * 1024), "MB")

3.608088493347168 MB


In [234]:
from google.colab import files

files.download("/content/champion_model.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [238]:
print("Model features:")
print(model_prev_train.booster_.feature_name())

print("\nVariables containing train/lag:")
print([x for x in globals() if "train" in x.lower() or "lag" in x.lower()])

Model features:
['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'past_segment_mean', 'day_of_week', 'month', 'is_weekend', 'previous_train_delay']

Variables containing train/lag:
['sample_train', 'train_ns', 'X_train_ns', 'y_train_ns', 'X_train_imp', 'y_train_imp', 'X_train_past', 'y_train_past', 'X_train_final', 'y_train_final', 'daily_train', 'latest_train_history', 'features_train_history', 'X_train_th', 'y_train_th', 'model_train_history', 'X_train_excess', 'y_train_excess', 'X_train_risk', 'y_train_risk', 'X_train_no_risk', 'y_train_no_risk', 'X_train_no_excess', 'y_train_no_excess', 'X_train_no_segment', 'y_train_no_segment', 'train_day_check', 'last_train_delay', 'features_prev_train', 'X_train_prev_train', 'X_val_prev_train', 'y_train_prev_train', 'y_val_prev_train', 'categorical_prev_train', 'target_prev_train', 'model_prev_train', 'train_cats', 'X_train_prev_predict', 'last_train_segment_delay', 'train_history_values', 'training

In [239]:
 print("Model type:", type(model_prev_train))
print("Number of features:", model_prev_train.booster_.num_feature())

Model type: <class 'lightgbm.sklearn.LGBMRegressor'>
Number of features: 10


In [240]:
print("Categorical features:")
print(model_prev_train.booster_.pandas_categorical)

Categorical features:
[[961, 962, 1007, 1008, 1023, 1024, 1025, 1026, 1027, 1028, 1031, 1032, 1091, 1103, 1104, 1139, 1140, 1155, 1156, 1163, 1164, 1165, 1166, 1167, 1168, 1171, 1172, 1181, 1183, 1185, 1186, 1203, 1211, 1212, 1303, 1307, 1315, 1317, 1319, 1321, 1323, 1351, 1353, 1355, 1357, 1359, 1361, 1365, 1367, 1369, 1401, 1435, 1436, 1437, 1445, 1446, 1461, 1462, 1487, 1488, 1581, 1587, 1589, 1591, 1592, 1595, 1596, 1663, 1664, 1665, 1666, 1667, 1668, 1701, 1702, 1707, 1708, 1883, 1904, 1905, 1906, 1919, 1920, 1921, 1922, 2001, 2023, 2024, 2121, 2122, 2131, 2132, 2133, 2134, 2185, 2186, 2187, 2188, 2197, 2198, 2199, 2200, 2249, 2310, 2391, 2392, 2393, 2394, 2397, 2398, 2525, 2526, 2563, 2564, 2569, 2570, 2575, 2576, 2813, 2814, 2831, 2832, 2837, 2838, 2839, 2840, 2847, 2848, 2897, 2898, 3007, 3008, 3043, 3044, 3047, 3048, 3109, 3110, 3132, 3172, 3183, 3190, 3191, 3204, 3206, 3225, 3226, 3229, 3230, 3239, 3240, 3241, 3242, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 3253, 3255, 

In [241]:
print("Categorical features:")
print(model_prev_train.booster_.pandas_categorical)

Categorical features:
[[961, 962, 1007, 1008, 1023, 1024, 1025, 1026, 1027, 1028, 1031, 1032, 1091, 1103, 1104, 1139, 1140, 1155, 1156, 1163, 1164, 1165, 1166, 1167, 1168, 1171, 1172, 1181, 1183, 1185, 1186, 1203, 1211, 1212, 1303, 1307, 1315, 1317, 1319, 1321, 1323, 1351, 1353, 1355, 1357, 1359, 1361, 1365, 1367, 1369, 1401, 1435, 1436, 1437, 1445, 1446, 1461, 1462, 1487, 1488, 1581, 1587, 1589, 1591, 1592, 1595, 1596, 1663, 1664, 1665, 1666, 1667, 1668, 1701, 1702, 1707, 1708, 1883, 1904, 1905, 1906, 1919, 1920, 1921, 1922, 2001, 2023, 2024, 2121, 2122, 2131, 2132, 2133, 2134, 2185, 2186, 2187, 2188, 2197, 2198, 2199, 2200, 2249, 2310, 2391, 2392, 2393, 2394, 2397, 2398, 2525, 2526, 2563, 2564, 2569, 2570, 2575, 2576, 2813, 2814, 2831, 2832, 2837, 2838, 2839, 2840, 2847, 2848, 2897, 2898, 3007, 3008, 3043, 3044, 3047, 3048, 3109, 3110, 3132, 3172, 3183, 3190, 3191, 3204, 3206, 3225, 3226, 3229, 3230, 3239, 3240, 3241, 3242, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 3253, 3255, 

In [242]:
print(model_prev_train.booster_.pandas_categorical)

[[961, 962, 1007, 1008, 1023, 1024, 1025, 1026, 1027, 1028, 1031, 1032, 1091, 1103, 1104, 1139, 1140, 1155, 1156, 1163, 1164, 1165, 1166, 1167, 1168, 1171, 1172, 1181, 1183, 1185, 1186, 1203, 1211, 1212, 1303, 1307, 1315, 1317, 1319, 1321, 1323, 1351, 1353, 1355, 1357, 1359, 1361, 1365, 1367, 1369, 1401, 1435, 1436, 1437, 1445, 1446, 1461, 1462, 1487, 1488, 1581, 1587, 1589, 1591, 1592, 1595, 1596, 1663, 1664, 1665, 1666, 1667, 1668, 1701, 1702, 1707, 1708, 1883, 1904, 1905, 1906, 1919, 1920, 1921, 1922, 2001, 2023, 2024, 2121, 2122, 2131, 2132, 2133, 2134, 2185, 2186, 2187, 2188, 2197, 2198, 2199, 2200, 2249, 2310, 2391, 2392, 2393, 2394, 2397, 2398, 2525, 2526, 2563, 2564, 2569, 2570, 2575, 2576, 2813, 2814, 2831, 2832, 2837, 2838, 2839, 2840, 2847, 2848, 2897, 2898, 3007, 3008, 3043, 3044, 3047, 3048, 3109, 3110, 3132, 3172, 3183, 3190, 3191, 3204, 3206, 3225, 3226, 3229, 3230, 3239, 3240, 3241, 3242, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 3253, 3255, 3256, 3257, 3258, 3259

In [243]:
import json
import os

cats = model_prev_train.booster_.pandas_categorical

print("Number of categorical columns:", len(cats))
print("Category counts:", [len(x) for x in cats])

Number of categorical columns: 3
Category counts: [3862, 4705, 4705]


In [244]:
print(model_prev_train.booster_.feature_name())
print()
print(model_prev_train.booster_.pandas_categorical)

['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'past_segment_mean', 'day_of_week', 'month', 'is_weekend', 'previous_train_delay']

[[961, 962, 1007, 1008, 1023, 1024, 1025, 1026, 1027, 1028, 1031, 1032, 1091, 1103, 1104, 1139, 1140, 1155, 1156, 1163, 1164, 1165, 1166, 1167, 1168, 1171, 1172, 1181, 1183, 1185, 1186, 1203, 1211, 1212, 1303, 1307, 1315, 1317, 1319, 1321, 1323, 1351, 1353, 1355, 1357, 1359, 1361, 1365, 1367, 1369, 1401, 1435, 1436, 1437, 1445, 1446, 1461, 1462, 1487, 1488, 1581, 1587, 1589, 1591, 1592, 1595, 1596, 1663, 1664, 1665, 1666, 1667, 1668, 1701, 1702, 1707, 1708, 1883, 1904, 1905, 1906, 1919, 1920, 1921, 1922, 2001, 2023, 2024, 2121, 2122, 2131, 2132, 2133, 2134, 2185, 2186, 2187, 2188, 2197, 2198, 2199, 2200, 2249, 2310, 2391, 2392, 2393, 2394, 2397, 2398, 2525, 2526, 2563, 2564, 2569, 2570, 2575, 2576, 2813, 2814, 2831, 2832, 2837, 2838, 2839, 2840, 2847, 2848, 2897, 2898, 3007, 3008, 3043, 3044, 3047, 3048, 3109,

In [245]:
for i, cats in enumerate(model_prev_train.booster_.pandas_categorical):
    print(f"Categorical column {i}: {len(cats)} categories")
    print(cats[:10])
    print()

Categorical column 0: 3862 categories
[961, 962, 1007, 1008, 1023, 1024, 1025, 1026, 1027, 1028]

Categorical column 1: 4705 categories
['AADR', 'AAG', 'AAL', 'AAM', 'AAR', 'AAS', 'AAY', 'AB', 'ABD', 'ABI']

Categorical column 2: 4705 categories
['AADR', 'AAG', 'AAL', 'AAM', 'AAR', 'AAS', 'AAY', 'AB', 'ABD', 'ABI']



In [246]:
import json

cats = model_prev_train.booster_.pandas_categorical

categories = {
    "train": cats[0],
    "current_station": cats[1],
    "next_station": cats[2]
}

path = "/content/station_categories.json"

with open(path, "w") as f:
    json.dump(categories, f, indent=2)

print("SUCCESS")
print("Saved:", path)
print("Train categories:", len(categories["train"]))
print("Current station categories:", len(categories["current_station"]))
print("Next station categories:", len(categories["next_station"]))

SUCCESS
Saved: /content/station_categories.json
Train categories: 3862
Current station categories: 4705
Next station categories: 4705


In [248]:
from google.colab import files

files.download("/content/station_categories.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [249]:
print(delay_df.columns.tolist())


['train', 'date', 'station', 'sch_arr', 'act_arr', 'arr_delay', 'sch_dep', 'act_dep', 'dep_delay', 'next_station', 'next_arr_delay', 'next_dep_delay', 'next_sch_arr']


In [250]:
print(delay_df.shape)
print(delay_df.columns.tolist())

(1282325, 13)
['train', 'date', 'station', 'sch_arr', 'act_arr', 'arr_delay', 'sch_dep', 'act_dep', 'dep_delay', 'next_station', 'next_arr_delay', 'next_dep_delay', 'next_sch_arr']


In [254]:
import os

print(os.path.exists("/content/ir_train.csv"))

True


In [255]:
import pandas as pd

train_df = pd.read_csv("/content/ir_train.csv")

print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())


Shape: (1500000, 45)
Columns: ['journey_id', 'train_number', 'train_type', 'departure_date', 'year', 'month', 'day_of_week', 'departure_hour', 'is_weekend', 'is_night_departure', 'is_peak_hour', 'is_festival_season', 'season', 'zone', 'zone_abbr', 'source_station_category', 'destination_station_category', 'distance_km', 'num_scheduled_stops', 'scheduled_travel_hours', 'track_doubled', 'is_hdn_route', 'traction_type', 'is_electrified', 'psr_count', 'is_circular_route', 'is_monsoon_season', 'is_fog_risk', 'fog_risk_score', 'zone_fog_index', 'zone_congestion_index', 'season_severity_score', 'loco_age_years', 'coach_age_years', 'has_lhb_coaches', 'is_rake_shared', 'maintenance_score', 'seat_utilisation_pct', 'is_overloaded', 'late_incoming_rake', 'is_special_train', 'route_historical_ontime_pct', 'primary_delay_cause', 'delay_minutes', 'is_delayed']


In [256]:
print(train_df[[
    "train_number",
    "departure_date",
    "source_station_category",
    "destination_station_category",
    "zone",
    "distance_km",
    "delay_minutes"
]].head())

   train_number departure_date source_station_category  \
0         12536     2023-09-18                       A   
1         12806     2020-04-09                      A1   
2         22416     2024-12-13                       E   
3         12250     2018-11-29                       C   
4         74588     2020-06-18                       C   

  destination_station_category                              zone  distance_km  \
0                            B       North Western Railway (NWR)          620   
1                            E        East Central Railway (ECR)         1423   
2                           A1       North Eastern Railway (NER)          449   
3                            D              Central Railway (CR)         1443   
4                            D  Northeast Frontier Railway (NFR)          219   

   delay_minutes  
0            110  
1              3  
2              5  
3            135  
4            154  


In [257]:
print(train_df["journey_id"].head(20).tolist())

['IR01717809', 'IR00931776', 'IR00096904', 'IR00047736', 'IR00726037', 'IR01774740', 'IR01270166', 'IR01634602', 'IR01115850', 'IR00731932', 'IR00101633', 'IR01845377', 'IR01796944', 'IR01623331', 'IR01207230', 'IR00294436', 'IR01384529', 'IR00121824', 'IR00948106', 'IR01816658']


In [258]:
print("Unique trains:", train_df["train_number"].nunique())
print(train_df["train_number"].unique()[:30])

Unique trains: 6293
[12536 12806 22416 12250 74588 19281 12550 74616 12664 11235 11576 12156
 51364 12331 74529 19886 11763 74580 12953 22250 11651 22877 74709 19997
 12107 74204 12899 12870 11815 12247]


In [259]:
print(
    train_df["train_number"]
    .value_counts()
    .describe()
)

count    6293.000000
mean      238.360083
std       196.724029
min         6.000000
25%       120.000000
50%       159.000000
75%       454.000000
max       676.000000
Name: count, dtype: float64


In [260]:
print(
    train_df.groupby("train_number")["distance_km"].first().describe()
)

count    6293.000000
mean      509.169395
std       327.070559
min        57.000000
25%       266.000000
50%       443.000000
75%       664.000000
max      2695.000000
Name: distance_km, dtype: float64


In [261]:
print(train_df[["train_number", "zone", "distance_km"]].drop_duplicates("train_number").head(20).to_string(index=False))

 train_number                              zone  distance_km
        12536       North Western Railway (NWR)          620
        12806        East Central Railway (ECR)         1423
        22416       North Eastern Railway (NER)          449
        12250              Central Railway (CR)         1443
        74588  Northeast Frontier Railway (NFR)          219
        19281         East Coast Railway (ECoR)          370
        12550  Northeast Frontier Railway (NFR)         1057
        74616              Central Railway (CR)          130
        12664              Central Railway (CR)          792
        11235        West Central Railway (WCR)          630
        11576       North Eastern Railway (NER)          574
        12156       South Western Railway (SWR)         1079
        51364       North Western Railway (NWR)          201
        12331        East Central Railway (ECR)         1661
        74529             Northern Railway (NR)          171
        19886       Nort

In [262]:
sample = train_df[train_df["train_number"] == 12536].copy()

print(sample[[
    "train_number",
    "departure_date",
    "distance_km",
    "scheduled_travel_hours",
    "delay_minutes"
]].head(20).to_string(index=False))

 train_number departure_date  distance_km  scheduled_travel_hours  delay_minutes
        12536     2023-09-18          620                   10.33            110
        12536     2019-11-15          521                    8.68              8
        12536     2024-07-31          463                    7.98             97
        12536     2024-06-30          427                    5.69            101
        12536     2018-09-03          585                    7.80            117
        12536     2021-06-08          670                   11.17            106
        12536     2023-11-20          363                    4.84              3
        12536     2024-07-05         1045                   14.51            134
        12536     2020-09-08          985                   16.42            138
        12536     2019-07-12          343                    4.57             83
        12536     2019-07-27         1675                   21.47             77
        12536     2024-09-10

In [264]:
print([x for x in globals() if "model" in x.lower()])

['model_ns', 'model_features', 'model_past', 'model_final', 'model_train_history', 'model_excess', 'model_change', 'model_risk', 'model_no_risk', 'model_no_excess', 'model_no_segment', 'model_prev_train', 'model_prev_segment']


In [265]:
for name in [
    "model_ns",
    "model_features",
    "model_past",
    "model_final",
    "model_train_history",
    "model_excess",
    "model_change",
    "model_risk",
    "model_no_risk",
    "model_no_excess",
    "model_no_segment",
    "model_prev_train",
    "model_prev_segment"
]:
    m = globals()[name]
    print("\n", name)
    print(type(m))


 model_ns
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_features
<class 'list'>

 model_past
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_final
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_train_history
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_excess
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_change
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_risk
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_no_risk
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_no_excess
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_no_segment
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_prev_train
<class 'lightgbm.sklearn.LGBMRegressor'>

 model_prev_segment
<class 'lightgbm.sklearn.LGBMRegressor'>


In [266]:
print("model_final features:")
print(model_final.feature_name_)

model_final features:
['train', 'current_station', 'next_station', 'current_arr_delay', 'scheduled_segment_minutes', 'month', 'day_of_week', 'is_weekend', 'past_segment_mean', 'past_segment_median', 'past_segment_std', 'past_segment_count']


In [267]:
print("\nmodel_final number of features:")
print(model_final.n_features_in_)


model_final number of features:
12


In [268]:
print(train_df.columns.tolist())

['journey_id', 'train_number', 'train_type', 'departure_date', 'year', 'month', 'day_of_week', 'departure_hour', 'is_weekend', 'is_night_departure', 'is_peak_hour', 'is_festival_season', 'season', 'zone', 'zone_abbr', 'source_station_category', 'destination_station_category', 'distance_km', 'num_scheduled_stops', 'scheduled_travel_hours', 'track_doubled', 'is_hdn_route', 'traction_type', 'is_electrified', 'psr_count', 'is_circular_route', 'is_monsoon_season', 'is_fog_risk', 'fog_risk_score', 'zone_fog_index', 'zone_congestion_index', 'season_severity_score', 'loco_age_years', 'coach_age_years', 'has_lhb_coaches', 'is_rake_shared', 'maintenance_score', 'seat_utilisation_pct', 'is_overloaded', 'late_incoming_rake', 'is_special_train', 'route_historical_ontime_pct', 'primary_delay_cause', 'delay_minutes', 'is_delayed']


In [269]:
[x for x in globals() if hasattr(globals()[x], "columns")]

['delay_df',
 'next_station_df',
 'sample',
 'train_ns',
 'val_ns',
 'X_train_ns',
 'X_val_ns',
 'importance',
 'segment_stats',
 'segment_history',
 'X_train_imp',
 'X_val_imp',
 'daily_segment',
 'latest_segment_history',
 'X_train_past',
 'X_val_past',
 'daily_segment_extra',
 'latest_extra_history',
 'X_train_final',
 'X_val_final',
 'daily_train',
 'latest_train_history',
 'X_train_th',
 'X_val_th',
 'error_analysis',
 'X_train_excess',
 'X_val_excess',
 'error_change',
 'zero_delay',
 'X_train_risk',
 'X_val_risk',
 'importance_risk',
 'X_train_no_risk',
 'X_val_no_risk',
 'X_train_no_excess',
 'X_val_no_excess',
 'X_train_no_segment',
 'X_val_no_segment',
 'bias_analysis',
 'worst',
 'train_day_check',
 'X_train_prev_train',
 'X_val_prev_train',
 'X_val_prev_predict',
 'X_train_prev_predict',
 'combined_test',
 'val_test',
 'comparison',
 'importance_prev',
 'X_train_prev_segment',
 'X_val_prev_segment',
 'train_df']

In [270]:
print(segment_stats.shape)
print(segment_stats.columns.tolist())
display(segment_stats.head())

(17317, 4)
['mean', 'median', 'std', 'count']


,mean,median,std,count
segment,,,,
DR->CSMT,18.926050,0.0,56.985143,1190
KUR->BBS,39.594737,9.0,99.518237,1140
JL->BSL,35.255526,13.0,83.936529,1131
BBS->KUR,70.667860,25.0,129.533806,1117
KYN->TNA,40.438739,7.0,89.022997,1110


In [271]:
# Reset the index so "segment" becomes a normal column
segment_stats_export = segment_stats.reset_index()

print(segment_stats_export.columns.tolist())
display(segment_stats_export.head())

['segment', 'mean', 'median', 'std', 'count']


,segment,mean,median,std,count
0,DR->CSMT,18.926050,0.0,56.985143,1190
1,KUR->BBS,39.594737,9.0,99.518237,1140
2,JL->BSL,35.255526,13.0,83.936529,1131
3,BBS->KUR,70.667860,25.0,129.533806,1117
4,KYN->TNA,40.438739,7.0,89.022997,1110


In [272]:
segment_stats_export.to_csv(
    "/content/segment_stats.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [273]:
print(segment_stats_export.dtypes)

segment     object
mean       float64
median     float64
std        float64
count        int64
dtype: object


In [274]:
import os
print(os.path.getsize("/content/segment_stats.csv"))

835254


In [275]:
print("STLIA->GWL" in segment_stats.index)
print("DR->CSMT" in segment_stats.index)

False
True


In [276]:
segments = set(segment_stats.index)

print("Total historical segments:", len(segments))

# Check the first few segments
print(list(segments)[:20])

Total historical segments: 17317
['GMR->DLN', 'CNO->MAO', 'RHN->KPV', 'CPP->VZM', 'BPL->NGP', 'JNO->HRG', 'BBK->CLJ', 'JRT->MVG', 'WANI->BUX', 'GDG->BNP', 'RAJP->KKW', 'BXN->BTE', 'BMGN->CGON', 'BAB->VGLJ', 'GNP->RGP', 'RURA->PNKD', 'SUR->TLT', 'KCG->MBNR', 'MTM->PAV', 'BLRR->CRLM']


In [277]:
for seg in ["DAA->DBA", "DBA->GWL", "ARI->GWL", "SLV->GWL", "STLIA->GWL"]:
    print(seg, "=>", seg in segments)

DAA->DBA => True
DBA->GWL => True
ARI->GWL => False
SLV->GWL => False
STLIA->GWL => False
